In [1]:
# ============================================================
# 024_rq_update_from_updates_and_meetings
# ============================================================
#
# Overview
# ----------------
# This notebook updates Research Questions (RQs) based on:
# (1) recent paper additions/updates from the Literature DB, and
# (2) meeting notes capturing explicit RQ change requests.
# It synthesizes signals from both sources using OpenAI,
# proposes new or revised RQs with versioning and evidence links,
# and writes proposals back to Notion in "Draft" status for human review.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - Notion Literature DB (recent papers by added_date and importance score)
#   - Notion Meeting DB (recent meeting notes with RQ change requests)
#   - Notion RQ DB (existing RQs for update candidates)
#   - env.txt (OPENAI_API_KEY, NOTION_TOKEN, DB IDs)
# Outputs:
#   - RQ proposal pages in Notion (Draft status, versioned, with evidence)
#   - Local JSON/CSV artifacts: update_bundle.json, rq_proposals.json, rq_proposals.csv
#   - Console preview table of proposals for human review
#
# Structure
# ----------------
# Cell 01: Environment setup and authentication
# Cell 02: Configuration and constants (schema, filters, paths)
# Cell 03: Notion client wrapper functions (query, create, update)
# Cell 04: OpenAI client wrapper functions (synthesis, proposal generation)
# Cell 05: Load recent paper updates from Literature DB
# Cell 06: Extract paper signals for RQ relevance
# Cell 07: Load recent meeting notes from Meeting DB
# Cell 08: Extract meeting signals and explicit RQ change requests
# Cell 09: Consolidate update bundle (papers + meetings)
# Cell 10: Generate RQ proposals using OpenAI (edits and new RQs)
# Cell 11: Versioning strategy and Notion write-back logic
# Cell 12: Human review helpers (preview table, dry-run mode)
# Cell 13: Write proposals to Notion (with dry_run switch)
# Cell 14: Save output artifacts (JSON and CSV)
# Cell 15: Summary and next steps
#
# Notes
# ----------------
# - All environment variables loaded from env.txt (not .env).
# - Versioning: create new RQ page linked to parent via "parent RQ" relation.
# - Draft status used by default; human reviews and promotes to Active.
# - Rate limiting and error handling included for Notion/OpenAI calls.
# - Artifacts saved to ./artifacts/day24/ with timestamp.
# - No secrets printed; validation checks confirm API connectivity.

In [3]:
# ============================================================
# Cell 01 — Environment setup and authentication
# ============================================================
# Overview:
#   Load environment variables from env.txt, initialize API clients
#   (Notion, OpenAI), validate credentials, and set up shared imports.
# Inputs / Outputs:
#   Inputs: env.txt (OPENAI_API_KEY, NOTION_TOKEN, database IDs)
#   Outputs: authenticated clients (notion_client, openai_client)
# Notes:
#   - No secrets printed; validation checks confirm connectivity.
#   - All subsequent cells rely on these initialized clients.

# --- Mandatory env loading ---
from dotenv import load_dotenv
import os
import sys
from datetime import datetime, timedelta
import json
import time

load_dotenv('env.txt')

# --- Runtime LLM configuration (given / assumed) ---
llm_provider = 'OpenAI'
llm_model = 'gpt-4o-mini'
llm_temperature = 0.0

# --- Shared imports ---
import pandas as pd
from typing import Dict, List, Any, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# --- Notion client initialization ---
try:
    from notion_client import Client as NotionClient
    NOTION_TOKEN = os.getenv('NOTION_TOKEN')
    if not NOTION_TOKEN:
        raise ValueError("NOTION_TOKEN not found in env.txt")
    notion_client = NotionClient(auth=NOTION_TOKEN)
    print("✓ Notion client initialized")
except Exception as e:
    print(f"✗ Notion client initialization failed: {e}")
    notion_client = None

# --- OpenAI client initialization ---
try:
    from openai import OpenAI
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY not found in env.txt")
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    print(f"✓ OpenAI client initialized (model: {llm_model})")
except Exception as e:
    print(f"✗ OpenAI client initialization failed: {e}")
    openai_client = None

# --- Database IDs from environment ---
LITERATURE_DB_ID = os.getenv('NOTION_LIT_DB_ID', '')
MEETING_DB_ID = os.getenv('NOTION_MTG_DB_ID', '')
RQ_DB_ID = os.getenv('NOTION_RQ_DB_ID', '')

if not all([LITERATURE_DB_ID, MEETING_DB_ID, RQ_DB_ID]):
    print("⚠ Warning: One or more database IDs missing in env.txt")
    print(f"  Literature DB: {'✓' if LITERATURE_DB_ID else '✗'}")
    print(f"  Meeting DB: {'✓' if MEETING_DB_ID else '✗'}")
    print(f"  RQ DB: {'✓' if RQ_DB_ID else '✗'}")
else:
    print("✓ All database IDs loaded")

# --- Artifact output directory ---
ARTIFACTS_DIR = './artifacts/day24/'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print(f"✓ Artifacts directory: {ARTIFACTS_DIR}")

# --- Timestamp for this run ---
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f"✓ Run timestamp: {RUN_TIMESTAMP}")

print("\n=== Environment setup complete ===")


✓ Notion client initialized
✓ OpenAI client initialized (model: gpt-4o-mini)
✓ All database IDs loaded
✓ Artifacts directory: ./artifacts/day24/
✓ Run timestamp: 20260120_055814

=== Environment setup complete ===


In [55]:
# ============================================================
# Cell 02 — Configuration and constants (schema, filters, paths)
# ============================================================
# Overview:
#   Define configuration constants for Notion schema mappings,
#   query filters, time windows, and output file paths.
# Inputs / Outputs:
#   Inputs: Environment variables (from Cell 01)
#   Outputs: Constants and filter configs for subsequent cells
# Notes:
#   - Schema property names must match actual Notion database structure
#   - Time windows configurable for recent papers and meetings
#   - All constants available to cells 03-15

# --- Time window configuration ---
# How far back to look for recent papers and meetings
RECENT_PAPERS_DAYS = 30  # Papers added/updated in last N days
RECENT_MEETINGS_DAYS = 14  # Meetings in last N days
MIN_PAPER_IMPORTANCE = 7  # Minimum importance score (0-10) to consider

print(f"Time windows: Papers={RECENT_PAPERS_DAYS}d, Meetings={RECENT_MEETINGS_DAYS}d")
print(f"Min paper importance: {MIN_PAPER_IMPORTANCE}")

# --- Notion Literature DB schema mapping ---
# Property names for querying and extracting paper data
LIT_SCHEMA = {
    "title": "Name",                 # title
    "created_time": "Created time",  # created_time 
    "authors_year": "Authors & Year",
    "tags": "Tags",
    "pdf_link": "PDF Link",
    "findings": "Findings",
    "core_idea": "Core Idea",
    "notes": "Notes",
    "methods": "Methods",
    "type": "Type",
    "source": "Source",
    "datasets": "Datasets",
    "papers_rel": "Papers",          # relation
}

# --- Notion Meeting DB schema mapping ---
MEETING_SCHEMA = {
    "title": "Name",
    "date": "Date",
    "tags": "Tags",
    "rq_mentions": "Papers",       # ここは “RQ mentions” ではなく、関連論文 relation
    "notes_primary": "Summary",    # まずSummaryを優先
    "notes_fallback": "Transcript",
    "interviewee": "Interviewee",
}


# --- Notion RQ DB schema mapping ---
RQ_SCHEMA = {
    "title": "Name",
    "status": "Status",
    "priority": "Priority",
    "tags": "Tags",
    "evidence": "Linked Paper",
    "rationale": "Rationale / Background",
    "approach": "Proposed Approach",
    "gap": "Gap Identified",
}

# --- Filter configuration for Notion queries ---
# Calculate cutoff dates
papers_cutoff = (datetime.now() - timedelta(days=RECENT_PAPERS_DAYS)).isoformat()
meetings_cutoff = (datetime.now() - timedelta(days=RECENT_MEETINGS_DAYS)).isoformat()

# Literature DB filter: recent papers with min importance
from datetime import datetime, timedelta, timezone

def iso_date_days_ago(days: int) -> str:
    dt = datetime.now(timezone.utc) - timedelta(days=days)
    return dt.isoformat()

LIT_FILTER = {
    "timestamp": "created_time",
    "created_time": {"on_or_after": iso_date_days_ago(RECENT_PAPERS_DAYS)}
}

# Meeting DB filter: recent meetings
MEETING_FILTER = {
    'property': MEETING_SCHEMA['date'],
    'date': {'on_or_after': meetings_cutoff}
}

# RQ DB filter: active RQs (candidates for updates)
RQ_FILTER = {
    "property": RQ_SCHEMA["status"],
    "status": {"does_not_equal": "Archived"}  
}

print(f"✓ Filters configured (papers cutoff: {papers_cutoff[:10]}, meetings: {meetings_cutoff[:10]})")

# --- OpenAI configuration ---
OPENAI_CONFIG = {
    'model': llm_model,
    'temperature': llm_temperature,
    'max_tokens': 2000,  # Per synthesis/proposal call
    'timeout': 30  # Seconds
}

print(f"✓ OpenAI config: {OPENAI_CONFIG['model']} (temp={OPENAI_CONFIG['temperature']})")

# --- Output file paths ---
UPDATE_BUNDLE_PATH = os.path.join(ARTIFACTS_DIR, f'update_bundle_{RUN_TIMESTAMP}.json')
PROPOSALS_JSON_PATH = os.path.join(ARTIFACTS_DIR, f'rq_proposals_{RUN_TIMESTAMP}.json')
PROPOSALS_CSV_PATH = os.path.join(ARTIFACTS_DIR, f'rq_proposals_{RUN_TIMESTAMP}.csv')

print(f"✓ Output paths configured:")
print(f"  Update bundle: {UPDATE_BUNDLE_PATH}")
print(f"  Proposals JSON: {PROPOSALS_JSON_PATH}")
print(f"  Proposals CSV: {PROPOSALS_CSV_PATH}")

# --- Rate limiting configuration ---
NOTION_RATE_LIMIT_DELAY = 0.35  # Seconds between Notion API calls
OPENAI_RATE_LIMIT_DELAY = 0.5  # Seconds between OpenAI API calls
MAX_RETRIES = 3  # Max retry attempts for transient failures

print(f"✓ Rate limits: Notion={NOTION_RATE_LIMIT_DELAY}s, OpenAI={OPENAI_RATE_LIMIT_DELAY}s")

# --- Dry run mode (global switch) ---
DRY_RUN = False  # Set to False to actually write to Notion
print(f"✓ Dry run mode: {DRY_RUN}")

print("\n=== Configuration complete ===")


Time windows: Papers=30d, Meetings=14d
Min paper importance: 7
✓ Filters configured (papers cutoff: 2025-12-21, meetings: 2026-01-06)
✓ OpenAI config: gpt-4o-mini (temp=0.0)
✓ Output paths configured:
  Update bundle: ./artifacts/day24/update_bundle_20260120_055814.json
  Proposals JSON: ./artifacts/day24/rq_proposals_20260120_055814.json
  Proposals CSV: ./artifacts/day24/rq_proposals_20260120_055814.csv
✓ Rate limits: Notion=0.35s, OpenAI=0.5s
✓ Dry run mode: False

=== Configuration complete ===


In [15]:
# ============================================================
# Cell 03 — Notion client wrapper functions (query, create, update)
# ============================================================
# Overview:
#   Provides robust wrapper functions for Notion API operations:
#   - query_database: Query with filters, pagination, rate limiting
#   - create_page: Create new pages with property validation
#   - update_page: Update existing pages with retry logic
#   - get_page: Retrieve single page by ID
#   All functions include error handling, rate limiting, and retry logic.
# Inputs / Outputs:
#   Inputs: notion_client (from Cell 01), database IDs, filters
#   Outputs: Parsed Notion data (dicts/lists), success/error indicators
# Notes:
#   - Respects NOTION_RATE_LIMIT_DELAY from Cell 02
#   - Retries transient failures (429, 503) up to MAX_RETRIES
#   - Returns structured data with consistent error handling

from typing import Dict, List, Any, Optional
import time

def query_database(
    database_id: str,
    filter_obj: Optional[Dict[str, Any]] = None,
    sorts: Optional[List[Dict[str, str]]] = None,
    page_size: int = 100
) -> List[Dict[str, Any]]:
    """
    Query a Notion database with pagination and rate limiting.
    
    Args:
        database_id: Notion database ID
        filter_obj: Notion filter object (optional)
        sorts: List of sort objects (optional)
        page_size: Results per page (max 100)
    
    Returns:
        List of page objects (full Notion API response format)
    """
    if not notion_client:
        print("✗ Notion client not initialized")
        return []
    
    results = []
    has_more = True
    start_cursor = None
    attempt = 0
    
    while has_more and attempt < MAX_RETRIES:
        try:
            query_params = {
                'database_id': database_id,
                'page_size': page_size
            }
            if filter_obj:
                query_params['filter'] = filter_obj
            if sorts:
                query_params['sorts'] = sorts
            if start_cursor:
                query_params['start_cursor'] = start_cursor
            
            response = notion_client.databases.query(**query_params)
            results.extend(response.get('results', []))
            
            has_more = response.get('has_more', False)
            start_cursor = response.get('next_cursor')
            
            time.sleep(NOTION_RATE_LIMIT_DELAY)
            attempt = 0  # Reset on success
            
        except Exception as e:
            attempt += 1
            if '429' in str(e) or '503' in str(e):  # Rate limit or service unavailable
                wait_time = NOTION_RATE_LIMIT_DELAY * (2 ** attempt)
                print(f"⚠ Rate limit/503, retrying in {wait_time:.1f}s (attempt {attempt}/{MAX_RETRIES})")
                time.sleep(wait_time)
            else:
                print(f"✗ Query error: {e}")
                break
    
    return results


def create_page(
    database_id: str,
    properties: Dict[str, Any],
    children: Optional[List[Dict[str, Any]]] = None
) -> Optional[Dict[str, Any]]:
    """
    Create a new page in a Notion database.
    
    Args:
        database_id: Target database ID
        properties: Page properties (Notion format)
        children: Optional list of block children
    
    Returns:
        Created page object or None on failure
    """
    if not notion_client:
        print("✗ Notion client not initialized")
        return None
    
    for attempt in range(MAX_RETRIES):
        try:
            create_params = {
                'parent': {'database_id': database_id},
                'properties': properties
            }
            if children:
                create_params['children'] = children
            
            page = notion_client.pages.create(**create_params)
            time.sleep(NOTION_RATE_LIMIT_DELAY)
            return page
            
        except Exception as e:
            if '429' in str(e) or '503' in str(e):
                wait_time = NOTION_RATE_LIMIT_DELAY * (2 ** (attempt + 1))
                print(f"⚠ Rate limit/503 on create, retrying in {wait_time:.1f}s")
                time.sleep(wait_time)
            else:
                print(f"✗ Create page error: {e}")
                return None
    
    print(f"✗ Create page failed after {MAX_RETRIES} attempts")
    return None


def update_page(
    page_id: str,
    properties: Dict[str, Any]
) -> Optional[Dict[str, Any]]:
    """
    Update properties of an existing Notion page.
    
    Args:
        page_id: Page ID to update
        properties: Properties to update (Notion format)
    
    Returns:
        Updated page object or None on failure
    """
    if not notion_client:
        print("✗ Notion client not initialized")
        return None
    
    for attempt in range(MAX_RETRIES):
        try:
            page = notion_client.pages.update(
                page_id=page_id,
                properties=properties
            )
            time.sleep(NOTION_RATE_LIMIT_DELAY)
            return page
            
        except Exception as e:
            if '429' in str(e) or '503' in str(e):
                wait_time = NOTION_RATE_LIMIT_DELAY * (2 ** (attempt + 1))
                print(f"⚠ Rate limit/503 on update, retrying in {wait_time:.1f}s")
                time.sleep(wait_time)
            else:
                print(f"✗ Update page error: {e}")
                return None
    
    print(f"✗ Update page failed after {MAX_RETRIES} attempts")
    return None


def get_page(page_id: str) -> Optional[Dict[str, Any]]:
    """
    Retrieve a single page by ID.
    
    Args:
        page_id: Notion page ID
    
    Returns:
        Page object or None on failure
    """
    if not notion_client:
        print("✗ Notion client not initialized")
        return None
    
    for attempt in range(MAX_RETRIES):
        try:
            page = notion_client.pages.retrieve(page_id=page_id)
            time.sleep(NOTION_RATE_LIMIT_DELAY)
            return page
            
        except Exception as e:
            if '429' in str(e) or '503' in str(e):
                wait_time = NOTION_RATE_LIMIT_DELAY * (2 ** (attempt + 1))
                print(f"⚠ Rate limit/503 on retrieve, retrying in {wait_time:.1f}s")
                time.sleep(wait_time)
            else:
                print(f"✗ Get page error: {e}")
                return None
    
    print(f"✗ Get page failed after {MAX_RETRIES} attempts")
    return None


def extract_property_value(page: Dict[str, Any], prop_name: str) -> Any:
    """
    Extract a property value from a Notion page object.
    Handles common property types: title, rich_text, number, select, date, relation, url.
    
    Args:
        page: Notion page object
        prop_name: Property name to extract
    
    Returns:
        Extracted value (type varies by property) or None
    """
    try:
        properties = page.get('properties', {})
        prop = properties.get(prop_name)
        if not prop:
            return None
        
        prop_type = prop.get('type')
        
        if prop_type == 'title':
            titles = prop.get('title', [])
            return titles[0].get('plain_text', '') if titles else ''
        
        elif prop_type == 'rich_text':
            texts = prop.get('rich_text', [])
            return ' '.join([t.get('plain_text', '') for t in texts])
        
        elif prop_type == 'number':
            return prop.get('number')
        
        elif prop_type == 'select':
            select = prop.get('select')
            return select.get('name') if select else None
        
        elif prop_type == 'multi_select':
            return [ms.get('name') for ms in prop.get('multi_select', [])]
        
        elif prop_type == 'date':
            date_obj = prop.get('date')
            return date_obj.get('start') if date_obj else None
        
        elif prop_type == 'relation':
            return [rel.get('id') for rel in prop.get('relation', [])]
        
        elif prop_type == 'url':
            return prop.get('url')
        
        else:
            return None
            
    except Exception as e:
        print(f"⚠ Property extraction error ({prop_name}): {e}")
        return None


print("✓ Notion wrapper functions defined:")
print("  - query_database (with pagination & retry)")
print("  - create_page (with retry logic)")
print("  - update_page (with retry logic)")
print("  - get_page (with retry logic)")
print("  - extract_property_value (common types)")
print("\n=== Notion wrapper functions ready ===")


✓ Notion wrapper functions defined:
  - query_database (with pagination & retry)
  - create_page (with retry logic)
  - update_page (with retry logic)
  - get_page (with retry logic)
  - extract_property_value (common types)

=== Notion wrapper functions ready ===


In [16]:
# ============================================================
# Cell 04 — OpenAI client wrapper functions (synthesis, proposal generation)
# ============================================================
# Overview:
#   Provides wrapper functions for OpenAI API operations:
#   - synthesize_update_signals: Analyze papers/meetings for RQ relevance
#   - generate_rq_proposals: Create RQ update/creation proposals
#   - validate_proposal: Check proposal structure and completeness
#   All functions include error handling, rate limiting, and retry logic.
# Inputs / Outputs:
#   Inputs: openai_client (from Cell 01), update bundles, existing RQs
#   Outputs: Structured synthesis results, RQ proposals (JSON)
# Notes:
#   - Respects OPENAI_RATE_LIMIT_DELAY from Cell 02
#   - Uses OPENAI_CONFIG (model, temperature, max_tokens)
#   - Returns structured JSON for downstream processing
#   - No API calls during generation; code will run later

from typing import Dict, List, Any, Optional
import json
import time


def synthesize_update_signals(
    papers: List[Dict[str, Any]],
    meetings: List[Dict[str, Any]],
    existing_rqs: List[Dict[str, Any]]
) -> Optional[Dict[str, Any]]:
    """
    Synthesize paper and meeting signals to identify RQ update opportunities.
    
    Args:
        papers: List of recent paper dicts (from Literature DB)
        meetings: List of recent meeting dicts (from Meeting DB)
        existing_rqs: List of existing active RQs
    
    Returns:
        Synthesis dict with:
        - paper_signals: List of {paper_id, relevance, suggested_rqs}
        - meeting_signals: List of {meeting_id, rq_requests, priority}
        - update_candidates: List of {rq_id, update_reason, confidence}
        - new_rq_suggestions: List of {topic, justification, priority}
    """
    if not openai_client:
        print("✗ OpenAI client not initialized")
        return None
    
    # Build synthesis prompt
    prompt = f"""You are analyzing research signals to identify Research Question (RQ) update opportunities.

EXISTING ACTIVE RQs ({len(existing_rqs)}):
{json.dumps(existing_rqs, indent=2)}

RECENT PAPERS ({len(papers)}):
{json.dumps(papers, indent=2)}

RECENT MEETINGS ({len(meetings)}):
{json.dumps(meetings, indent=2)}

Analyze these inputs and provide:
1. Paper signals: Which papers are relevant to which RQs? Why?
2. Meeting signals: Are there explicit RQ change requests in meetings?
3. Update candidates: Which existing RQs should be updated/revised?
4. New RQ suggestions: Should any new RQs be created based on these signals?

Return ONLY valid JSON matching this schema:
{{
  "paper_signals": [
    {{"paper_id": "...", "paper_title": "...", "relevance": "high|medium|low", "suggested_rqs": ["rq_id1", ...], "rationale": "..."}}
  ],
  "meeting_signals": [
    {{"meeting_id": "...", "meeting_title": "...", "rq_requests": [{{"rq_id": "...", "action": "update|create|deprecate", "reason": "..."}}], "priority": "high|medium|low"}}
  ],
  "update_candidates": [
    {{"rq_id": "...", "rq_title": "...", "update_reason": "...", "confidence": "high|medium|low", "evidence_ids": ["paper_id1", "meeting_id2", ...]}}
  ],
  "new_rq_suggestions": [
    {{"topic": "...", "justification": "...", "priority": "high|medium|low", "evidence_ids": ["..."]}}
  ]
}}
"""
    
    for attempt in range(MAX_RETRIES):
        try:
            response = openai_client.chat.completions.create(
                model=OPENAI_CONFIG['model'],
                messages=[
                    {"role": "system", "content": "You are a research synthesis assistant. Return only valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                temperature=OPENAI_CONFIG['temperature'],
                max_tokens=OPENAI_CONFIG['max_tokens'],
                timeout=OPENAI_CONFIG['timeout']
            )
            
            content = response.choices[0].message.content.strip()
            # Remove markdown fences if present
            if content.startswith('```'):
                content = '\n'.join(content.split('\n')[1:-1])
            
            synthesis = json.loads(content)
            time.sleep(OPENAI_RATE_LIMIT_DELAY)
            return synthesis
            
        except json.JSONDecodeError as e:
            print(f"✗ JSON decode error in synthesis (attempt {attempt + 1}): {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(OPENAI_RATE_LIMIT_DELAY * 2)
            else:
                return None
                
        except Exception as e:
            if 'rate_limit' in str(e).lower() or '429' in str(e):
                wait_time = OPENAI_RATE_LIMIT_DELAY * (2 ** (attempt + 1))
                print(f"⚠ OpenAI rate limit, retrying in {wait_time:.1f}s")
                time.sleep(wait_time)
            else:
                print(f"✗ Synthesis error: {e}")
                return None
    
    print(f"✗ Synthesis failed after {MAX_RETRIES} attempts")
    return None


def generate_rq_proposals(
    synthesis: Dict[str, Any],
    existing_rqs: List[Dict[str, Any]]
) -> Optional[List[Dict[str, Any]]]:
    """
    Generate concrete RQ proposals (updates and new RQs) based on synthesis.
    
    Args:
        synthesis: Output from synthesize_update_signals
        existing_rqs: List of existing active RQs
    
    Returns:
        List of proposal dicts, each containing:
        - action: 'update' or 'create'
        - rq_id: Existing RQ ID (for updates) or None (for new)
        - title: Proposed RQ title/question
        - description: Detailed description
        - rationale: Why this proposal is needed
        - evidence_ids: List of paper/meeting IDs supporting this
        - priority: 'High', 'Medium', 'Low'
        - confidence: 'high', 'medium', 'low'
    """
    if not openai_client:
        print("✗ OpenAI client not initialized")
        return None
    
    # Build proposal generation prompt
    prompt = f"""You are generating concrete Research Question (RQ) proposals based on synthesis.

SYNTHESIS RESULTS:
{json.dumps(synthesis, indent=2)}

EXISTING RQs (for updates):
{json.dumps(existing_rqs, indent=2)}

Generate specific RQ proposals:
1. For each update_candidate: propose revised RQ text with clear changes
2. For each new_rq_suggestion: propose complete new RQ with title and description
3. Prioritize based on evidence strength and strategic importance
4. Link each proposal to supporting evidence (paper/meeting IDs)

Return ONLY valid JSON as a list of proposals:
[
  {{
    "action": "update",
    "rq_id": "existing_rq_id",
    "title": "Revised RQ question text",
    "description": "Detailed description of the revised RQ, including what changed and why",
    "rationale": "Why this update is needed (reference specific papers/meetings)",
    "evidence_ids": ["paper_id1", "meeting_id2"],
    "priority": "High|Medium|Low",
    "confidence": "high|medium|low",
    "version_increment": "minor|major"
  }},
  {{
    "action": "create",
    "rq_id": null,
    "title": "New RQ question text",
    "description": "Detailed description of the new RQ and its scope",
    "rationale": "Why this new RQ is needed",
    "evidence_ids": ["paper_id3"],
    "priority": "High|Medium|Low",
    "confidence": "high|medium|low"
  }}
]
"""
    
    for attempt in range(MAX_RETRIES):
        try:
            response = openai_client.chat.completions.create(
                model=OPENAI_CONFIG['model'],
                messages=[
                    {"role": "system", "content": "You are a research question proposal generator. Return only valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                temperature=OPENAI_CONFIG['temperature'],
                max_tokens=OPENAI_CONFIG['max_tokens'],
                timeout=OPENAI_CONFIG['timeout']
            )
            
            content = response.choices[0].message.content.strip()
            # Remove markdown fences if present
            if content.startswith('```'):
                content = '\n'.join(content.split('\n')[1:-1])
            
            proposals = json.loads(content)
            
            # Validate proposals structure
            if not isinstance(proposals, list):
                raise ValueError("Proposals must be a list")
            
            for p in proposals:
                if not validate_proposal(p):
                    raise ValueError(f"Invalid proposal structure: {p}")
            
            time.sleep(OPENAI_RATE_LIMIT_DELAY)
            return proposals
            
        except json.JSONDecodeError as e:
            print(f"✗ JSON decode error in proposals (attempt {attempt + 1}): {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(OPENAI_RATE_LIMIT_DELAY * 2)
            else:
                return None
                
        except Exception as e:
            if 'rate_limit' in str(e).lower() or '429' in str(e):
                wait_time = OPENAI_RATE_LIMIT_DELAY * (2 ** (attempt + 1))
                print(f"⚠ OpenAI rate limit, retrying in {wait_time:.1f}s")
                time.sleep(wait_time)
            else:
                print(f"✗ Proposal generation error: {e}")
                return None
    
    print(f"✗ Proposal generation failed after {MAX_RETRIES} attempts")
    return None


def validate_proposal(proposal: Dict[str, Any]) -> bool:
    """
    Validate that a proposal has required fields and valid values.
    
    Args:
        proposal: Proposal dict to validate
    
    Returns:
        True if valid, False otherwise
    """
    required_fields = ['action', 'title', 'description', 'rationale', 'priority', 'confidence']
    
    for field in required_fields:
        if field not in proposal:
            print(f"⚠ Missing required field: {field}")
            return False
    
    # Validate action
    if proposal['action'] not in ['update', 'create']:
        print(f"⚠ Invalid action: {proposal['action']}")
        return False
    
    # Validate rq_id for updates
    if proposal['action'] == 'update' and not proposal.get('rq_id'):
        print("⚠ Update action requires rq_id")
        return False
    
    # Validate priority
    if proposal['priority'] not in ['High', 'Medium', 'Low']:
        print(f"⚠ Invalid priority: {proposal['priority']}")
        return False
    
    # Validate confidence
    if proposal['confidence'] not in ['high', 'medium', 'low']:
        print(f"⚠ Invalid confidence: {proposal['confidence']}")
        return False
    
    # Validate evidence_ids (should be list)
    if 'evidence_ids' in proposal and not isinstance(proposal['evidence_ids'], list):
        print("⚠ evidence_ids must be a list")
        return False
    
    return True


print("✓ OpenAI wrapper functions defined:")
print("  - synthesize_update_signals (analyze papers/meetings)")
print("  - generate_rq_proposals (create proposals)")
print("  - validate_proposal (structure validation)")
print("\n=== OpenAI wrapper functions ready ===")


✓ OpenAI wrapper functions defined:
  - synthesize_update_signals (analyze papers/meetings)
  - generate_rq_proposals (create proposals)
  - validate_proposal (structure validation)

=== OpenAI wrapper functions ready ===


In [20]:
# ============================================================
# Cell 05: Load recent paper updates from Literature DB
# ============================================================
# Overview:
#   
# Inputs / Outputs:
#   
# Notes:
#   

import time
import requests
NOTION_VERSION = os.getenv("NOTION_VERSION")  
NOTION_HEADERS = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Notion-Version": NOTION_VERSION,
    "Content-Type": "application/json",
}
NOTION_API_BASE = "https://api.notion.com/v1"

def notion_post(path: str, payload: dict, timeout: int = 30) -> dict:
    url = f"{NOTION_API_BASE}{path}"
    r = requests.post(url, headers=NOTION_HEADERS, json=payload, timeout=timeout)
    if r.status_code >= 300:
        raise RuntimeError(f"Notion POST {path} failed: {r.status_code} {r.text}")
    return r.json()

def query_database(database_id: str, filter_obj: dict = None, sorts: list = None, page_size: int = 100, max_pages: int = 20, sleep_s: float = 0.2):
    """
    Returns list of Notion page objects from a database query.
    """
    results = []
    cursor = None
    pages_fetched = 0

    while True:
        payload = {"page_size": page_size}
        if filter_obj:
            payload["filter"] = filter_obj
        if sorts:
            payload["sorts"] = sorts
        if cursor:
            payload["start_cursor"] = cursor

        data = notion_post(f"/databases/{database_id}/query", payload)
        batch = data.get("results", [])
        results.extend(batch)

        pages_fetched += 1
        if not data.get("has_more"):
            break
        cursor = data.get("next_cursor")
        if not cursor:
            break
        if pages_fetched >= max_pages:
            print(f"⚠ Reached max_pages={max_pages}. Returning partial results: {len(results)}")
            break

        time.sleep(sleep_s)

    return results


print("\n=== Loading recent papers from Literature DB ===")
print(f"Database ID: {LITERATURE_DB_ID[:8]}...")
print(f"Filter: Added in last {RECENT_PAPERS_DAYS} days, importance >= {MIN_PAPER_IMPORTANCE}")

# Query Literature DB with configured filter
recent_paper_pages = query_database(
    database_id=LITERATURE_DB_ID,
    filter_obj=LIT_FILTER,
    sorts=[{"timestamp": "created_time", "direction": "descending"}]
)


print(f"✓ Retrieved {len(recent_paper_pages)} paper pages")

# Extract structured paper data
recent_papers = []

for page in recent_paper_pages:
    try:
        paper_id = page['id']
        
        # Extract all relevant properties
        paper_data = {
            "paper_id": paper_id,
            "title": extract_property_value(page, LIT_SCHEMA["title"]) or "Untitled",
            "created_time": page.get("created_time"),  # Notion top-level
            "authors_year": extract_property_value(page, LIT_SCHEMA["authors_year"]) or "",
            "tags": extract_property_value(page, LIT_SCHEMA["tags"]) or [],
            "pdf_link": extract_property_value(page, LIT_SCHEMA["pdf_link"]) or "",
            "core_idea": extract_property_value(page, LIT_SCHEMA["core_idea"]) or "",
            "findings": extract_property_value(page, LIT_SCHEMA["findings"]) or "",
            "methods": extract_property_value(page, LIT_SCHEMA["methods"]) or "",
            "notes": extract_property_value(page, LIT_SCHEMA["notes"]) or "",
            "type": extract_property_value(page, LIT_SCHEMA["type"]) or "",
            "source": extract_property_value(page, LIT_SCHEMA["source"]) or "",
            "datasets": extract_property_value(page, LIT_SCHEMA["datasets"]) or "",
        }

        
        recent_papers.append(paper_data)
        
    except Exception as e:
        print(f"⚠ Error extracting paper data (page {page.get('id', 'unknown')}): {e}")
        continue

print(f"✓ Extracted data for {len(recent_papers)} papers")

# Display summary statistics (recency-only)
if recent_papers:
    # created_time を datetime に寄せて扱う（文字列のままでも動くが、整形のため）
    def _to_dt(x):
        try:
            # Notion created_time is ISO8601 like "2026-01-20T..."
            return datetime.fromisoformat(x.replace("Z", "+00:00")) if isinstance(x, str) else None
        except Exception:
            return None

    created_list = [_to_dt(p.get("created_time")) for p in recent_papers if p.get("created_time")]
    created_list = [d for d in created_list if d is not None]

    if created_list:
        newest = max(created_list)
        oldest = min(created_list)
        print(f"  Date range (created_time): {oldest.date()} → {newest.date()}")
    else:
        print("  Date range (created_time): (unavailable)")

    # Tag distribution (top 10)
    from collections import Counter
    tag_counter = Counter()
    for p in recent_papers:
        for t in (p.get("tags") or []):
            tag_counter[t] += 1

    if tag_counter:
        print("  Top tags:")
        for tag, cnt in tag_counter.most_common(10):
            print(f"    - {tag}: {cnt}")
    else:
        print("  Top tags: (none)")

    # Preview latest 5 papers (by created_time desc)
    def _sort_key(p):
        dt = _to_dt(p.get("created_time"))
        # Noneは最後に回す
        return dt or datetime.min.replace(tzinfo=timezone.utc)

    sorted_papers = sorted(recent_papers, key=_sort_key, reverse=True)

    print("\n  Latest 5 papers (by created_time):")
    for i, paper in enumerate(sorted_papers[:5], 1):
        ct = paper.get("created_time", "")
        title = (paper.get("title") or "Untitled").strip()
        print(f"    {i}. {ct[:10]}  {title[:80]}")
else:
    print("  No papers found in the recent window")

print(f"\n✓ Recent papers loaded: {len(recent_papers)} total")
print("=== Paper loading complete ===")




=== Loading recent papers from Literature DB ===
Database ID: 2a98e0e4...
Filter: Added in last 30 days, importance >= 7
✓ Retrieved 33 paper pages
✓ Extracted data for 33 papers
  Date range (created_time): 2026-01-02 → 2026-01-19
  Top tags:
    - Venture Capital: 9
    - Governance: 3
    - entrepreneurship: 3
    - Regulation: 3
    - Corporate Finance: 2
    - Investment: 2
    - economic growth: 2
    - Policy Analysis: 2
    - India: 2
    - Corporate Venture Capital: 2

  Latest 5 papers (by created_time):
    1. 2026-01-19  Empirical Essays on Labor Regulation, Geopolitical Shocks, and Investment
    2. 2026-01-19  Who Govern Innovation Districts? A Grounded Investigation Through the Lens of Ju
    3. 2026-01-19  Sovereign Wealth Funds and Economic Growth in High Income Countries
    4. 2026-01-19  Analisis Kontra Naratif Kebijakan Pembentukan Danantara untuk Mendukung Ketahana
    5. 2026-01-19  Role of Alternative Investment Fund in Financing Startups

✓ Recent papers loade

In [24]:
# ============================================================
# Cell 06 — Extract paper signals for RQ relevance
# ============================================================
# Overview:
#   Analyze recent papers to extract signals relevant to existing RQs.
#   Identifies which papers inform which RQs and why, creating a
#   structured mapping for downstream synthesis.
# Inputs / Outputs:
#   Inputs: recent_papers (from Cell 05), RQ_DB_ID, RQ_SCHEMA
#   Outputs: paper_signals (list of dicts with paper-RQ mappings)
# Notes:
#   - Loads active RQs for comparison
#   - Creates lightweight relevance mapping (defers full synthesis to Cell 10)
#   - Stores paper metadata for evidence linking
#   - No OpenAI calls here; pure structural extraction

print("\n=== Extracting paper signals for RQ relevance ===")

# Step 1: Load active RQs from RQ DB
print("Loading active RQs from RQ database...")
active_rq_pages = query_database(
    database_id=RQ_DB_ID,
    filter_obj=RQ_FILTER,
    sorts=[{"timestamp": "last_edited_time", "direction": "descending"}]
)

print(f"✓ Retrieved {len(active_rq_pages)} active RQ pages")

# Step 2: Extract structured RQ data
active_rqs = []

for page in active_rq_pages:
    try:
        rq_id = page['id']
        
        rq_data = {
            "rq_id": rq_id,
            "title": extract_property_value(page, RQ_SCHEMA["title"]) or "Untitled RQ",
            "status": extract_property_value(page, RQ_SCHEMA["status"]) or "Unknown",
            "priority": extract_property_value(page, RQ_SCHEMA["priority"]) or "Unspecified",
            "tags": extract_property_value(page, RQ_SCHEMA["tags"]) or [],
            "evidence_ids": extract_property_value(page, RQ_SCHEMA["evidence"]) or [],
            "rationale": extract_property_value(page, RQ_SCHEMA["rationale"]) or "",
            "approach": extract_property_value(page, RQ_SCHEMA["approach"]) or "",
            "gap_identified": extract_property_value(page, RQ_SCHEMA["gap"]) or "",
            # Notion system timestamps (always available)
            "created_time": page.get("created_time"),
            "last_edited_time": page.get("last_edited_time"),
        }
        
        active_rqs.append(rq_data)
        
    except Exception as e:
        print(f"⚠ Error extracting RQ data (page {page.get('id', 'unknown')}): {e}")
        continue

print(f"✓ Extracted data for {len(active_rqs)} active RQs")

# Step 3: Build paper signals structure
# Each signal maps a paper to potentially relevant RQs based on:
# - Keyword/tag overlap
# - Topical alignment (summary/findings content)
# - Existing evidence links

paper_signals = []

for paper in recent_papers:
    paper_tags_lower = [str(tag).lower() for tag in (paper.get("tags") or [])]

    signal = {
        "paper_id": paper.get("paper_id"),
        "paper_title": paper.get("title", "Untitled"),
        "created_time": paper.get("created_time"),
        "potentially_relevant_rqs": [],
        "metadata": {
            "core_idea": (paper.get("core_idea") or "")[:300],
            "findings": (paper.get("findings") or "")[:300],
            "tags": paper.get("tags") or [],
            "authors_year": paper.get("authors_year") or "",
            "pdf_link": paper.get("pdf_link") or "",
        }
    }

    for rq in active_rqs:
        rq_tags_lower = [str(tag).lower() for tag in (rq.get("tags") or [])]
        tag_overlap = sorted(list(set(paper_tags_lower) & set(rq_tags_lower)))

        evidence_ids = set(rq.get("evidence_ids") or [])

        # relation が page_id リストなら paper_id と一致で拾える
        if tag_overlap or (paper.get("paper_id") in evidence_ids):
            signal["potentially_relevant_rqs"].append({
                "rq_id": rq.get("rq_id"),
                "rq_title": rq.get("title", "Untitled RQ"),
                "reason": f"Tag overlap: {tag_overlap}" if tag_overlap else "Already linked as evidence"
            })

    paper_signals.append(signal)

print(f"✓ Built {len(paper_signals)} paper signals")


# Step 4: Summary statistics
papers_with_rq_links = sum(1 for s in paper_signals if s['potentially_relevant_rqs'])
print(f"  Papers with potential RQ links: {papers_with_rq_links}/{len(paper_signals)}")

if papers_with_rq_links > 0:
    print("\n  Sample paper-RQ mappings:")
    for signal in [s for s in paper_signals if s['potentially_relevant_rqs']][:3]:
        print(f"    Paper: {signal['paper_title'][:50]}...")
        for link in signal['potentially_relevant_rqs'][:2]:
            print(f"      → RQ: {link['rq_title'][:50]}... ({link['reason']})")

print("\n✓ Paper signals extracted")
print("=== Paper signal extraction complete ===")



=== Extracting paper signals for RQ relevance ===
Loading active RQs from RQ database...
✓ Retrieved 30 active RQ pages
✓ Extracted data for 30 active RQs
✓ Built 33 paper signals
  Papers with potential RQ links: 14/33

  Sample paper-RQ mappings:
    Paper: Empirical Essays on Labor Regulation, Geopolitical...
      → RQ: 日本のディープテックスタートアップにおける資金不足問題は、どのような政策設計や投資家の関与によって解... (Tag overlap: ['investment'])
    Paper: Role of Alternative Investment Fund in Financing S...
      → RQ: 政府系VCがファイナンシャルリターンを重視する起業家の成功に与える影響とそのメカニズムは何か？... (Tag overlap: ['entrepreneurship'])
      → RQ: 日本のスタートアップがシリコンバレーのモデルを模倣せずに独自の成長モデルを確立するための制度設計とは... (Tag overlap: ['entrepreneurship'])
    Paper: Venture Capital vs. Corporate Venture Capital: Dif...
      → RQ: 日本のディープテックスタートアップにおける資金不足問題は、どのような政策設計や投資家の関与によって解... (Tag overlap: ['venture capital'])
      → RQ: クロスボーダー展開を目指すスタートアップにおいて、企業連携が現地市場適応と資金調達に及ぼす相乗効果の... (Tag overlap: ['venture capital'])

✓ Paper signals extracted
=== Paper signal extraction c

In [32]:
# ============================================================
# Cell 07 — Load recent meeting notes from Meeting DB
# ============================================================
# Overview:
#   Query the Notion Meeting DB for recent meeting notes that may
#   contain explicit RQ change requests or strategic discussions.
#   Extract meeting metadata and full notes for downstream analysis.
# Inputs / Outputs:
#   Inputs: MEETING_DB_ID, MEETING_FILTER, MEETING_SCHEMA (from Cell 02)
#   Outputs: recent_meetings (list of meeting dicts with notes)
# Notes:
#   - Uses query_database wrapper from Cell 03
#   - Filters by meeting date from Cell 02 config
#   - Extracts full notes content for RQ request parsing
#   - Empty result is valid (no recent meetings)

print("\n=== Loading recent meetings from Meeting DB ===")
print(f"Database ID: {MEETING_DB_ID[:8]}...")
print(f"Filter: Meetings in last {RECENT_MEETINGS_DAYS} days")

# Query Meeting DB with configured filter
recent_meeting_pages = query_database(
    database_id=MEETING_DB_ID,
    filter_obj=MEETING_FILTER,
    sorts=[{'property': MEETING_SCHEMA['date'], 'direction': 'descending'}]
)

print(f"✓ Retrieved {len(recent_meeting_pages)} meeting pages")

# Extract structured meeting data
recent_meetings = []

for page in recent_meeting_pages:
    try:
        meeting_id = page['id']
        
        # Extract notes from Summary first, fallback to Transcript
        notes_text = extract_property_value(page, MEETING_SCHEMA["notes_primary"]) or ""
        if not notes_text:
            notes_text = extract_property_value(page, MEETING_SCHEMA["notes_fallback"]) or ""
        
        meeting_data = {
            "meeting_id": meeting_id,
            "title": extract_property_value(page, MEETING_SCHEMA["title"]) or "Untitled Meeting",
            "date": extract_property_value(page, MEETING_SCHEMA["date"]),
            "notes": notes_text,  # ← Summary/Transcript を notes として統一
            "interviewee": extract_property_value(page, MEETING_SCHEMA["interviewee"]) or "",
            "tags": extract_property_value(page, MEETING_SCHEMA["tags"]) or [],
            "papers": extract_property_value(page, MEETING_SCHEMA["rq_mentions"]) or [],  # relation page ids
        }

        
        recent_meetings.append(meeting_data)
        
    except Exception as e:
        print(f"⚠ Error extracting meeting data (page {page.get('id', 'unknown')}): {e}")
        continue

print(f"✓ Extracted data for {len(recent_meetings)} meetings")

# Display summary statistics
if recent_meetings:
    # Count meetings with linked papers (relation field)
    with_papers = sum(1 for m in recent_meetings if (m.get('papers') or []))
    print(f"  Meetings with linked papers: {with_papers}")

    # Count meetings with substantial notes
    with_notes = sum(1 for m in recent_meetings if len(m.get('notes') or '') > 100)
    print(f"  Meetings with substantial notes (>100 chars): {with_notes}")

    # Preview recent meetings
    print("\n  Recent meetings:")
    for i, meeting in enumerate(recent_meetings[:3], 1):
        date_str = meeting.get('date', '')[:10] if meeting.get('date') else 'No date'
        notes_preview = (meeting.get('notes') or '')[:60].replace('\n', ' ')
        notes_preview = notes_preview if notes_preview else '(no notes)'
        print(f"    {i}. [{date_str}] {str(meeting.get('title',''))[:40]}...")
        print(f"       Notes preview: {notes_preview}...")
        if meeting.get('papers'):
            print(f"       Linked papers: {len(meeting.get('papers'))} linked")
else:
    print("  No meetings found matching criteria")


print(f"\n✓ Recent meetings loaded: {len(recent_meetings)} total")
print("=== Meeting loading complete ===")



=== Loading recent meetings from Meeting DB ===
Database ID: 2a98e0e4...
Filter: Meetings in last 14 days
✓ Retrieved 1 meeting pages
✓ Extracted data for 1 meetings
  Meetings with linked papers: 0
  Meetings with substantial notes (>100 chars): 1

  Recent meetings:
    1. [2026-01-14] CVCの進化とベストプラクティス：Agile対応パネル討論...
       Notes preview: JVCA主催のパネルディスカッションで、CVC（コーポレートベンチャーキャピタル）の進化とベストプラクティスについて議論...

✓ Recent meetings loaded: 1 total
=== Meeting loading complete ===


In [34]:
# ============================================================
# Cell 08 — Extract meeting signals and explicit RQ change requests
# ============================================================
# Overview:
#   Parse meeting notes to identify explicit RQ change requests,
#   strategic discussions, and action items related to research questions.
#   Creates structured meeting signals for downstream synthesis.
# Inputs / Outputs:
#   Inputs: recent_meetings (from Cell 07), active_rqs (from Cell 06)
#   Outputs: meeting_signals (list of dicts with RQ requests and context)
# Notes:
#   - Parses notes for RQ-related keywords and action items
#   - Identifies explicit requests (create, update, deprecate RQs)
#   - Links meetings to mentioned RQs via relation field or text analysis
#   - Assigns priority based on context and attendees
import json
import re
import time

def extract_json_object(text: str) -> dict:
    """
    Tries hard to extract a JSON object from model output.
    Accepts outputs like:
    - pure JSON
    - fenced ```json ... ```
    - extra text before/after JSON
    """
    if text is None:
        raise ValueError("Empty response text")

    # Remove code fences if present
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    # Try direct parse
    try:
        return json.loads(cleaned)
    except Exception:
        pass

    # Extract first {...} block
    m = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if not m:
        raise ValueError(f"Could not find JSON object in output: {cleaned[:200]}...")

    candidate = m.group(0)
    return json.loads(candidate)
    
def compact_text(s: str, max_chars: int = 1200) -> str:
    s = (s or "").strip()
    return s if len(s) <= max_chars else s[:max_chars] + "..."

meeting_evidence = []
for m in recent_meetings:
    meeting_evidence.append({
        "meeting_id": m["meeting_id"],
        "date": m.get("date"),
        "title": m.get("title"),
        "tags": m.get("tags") or [],
        "papers": m.get("papers") or [],
        "notes": compact_text(m.get("notes") or "", 2000),  # Summary/Transcript merged
    })

paper_evidence = []
for p in recent_papers:
    paper_evidence.append({
        "paper_id": p.get("paper_id"),
        "title": p.get("title"),
        "created_time": p.get("created_time"),
        "tags": p.get("tags") or [],
        "core_idea": compact_text(p.get("core_idea") or "", 800),
        "findings": compact_text(p.get("findings") or "", 800),
        "pdf_link": p.get("pdf_link") or "",
    })

evidence_bundle = {
    "meetings": meeting_evidence,
    "papers": paper_evidence,
}

print("✓ Built evidence_bundle")
print(f"  meetings: {len(evidence_bundle['meetings'])}")
print(f"  papers: {len(evidence_bundle['papers'])}")

def select_relevant_evidence_for_rq(rq: dict, bundle: dict, max_papers: int = 8, max_meetings: int = 3):
    rq_tags = set([str(t).lower() for t in (rq.get("tags") or [])])

    # papers: tag overlap
    scored_papers = []
    for p in bundle["papers"]:
        p_tags = set([str(t).lower() for t in (p.get("tags") or [])])
        overlap = len(rq_tags & p_tags) if rq_tags else 0
        if overlap > 0:
            scored_papers.append((overlap, p))
    scored_papers.sort(key=lambda x: x[0], reverse=True)
    papers = [p for _, p in scored_papers[:max_papers]]

    # meetings: if meeting references any selected paper ids, include it
    selected_paper_ids = set([p["paper_id"] for p in papers if p.get("paper_id")])
    meetings = []
    for m in bundle["meetings"]:
        m_papers = set(m.get("papers") or [])
        if selected_paper_ids and (m_papers & selected_paper_ids):
            meetings.append(m)
    # fallback: latest meeting as context
    if not meetings and bundle["meetings"]:
        meetings = bundle["meetings"][:1]

    return {"papers": papers, "meetings": meetings[:max_meetings]}

def llm_rq_decision(
    rq: dict,
    relevant: dict,
    model: str = "gpt-4.1-mini",
    max_retries: int = 2,
    sleep_s: float = 1.0
) -> dict:
    """
    Calls OpenAI to decide KEEP/UPDATE/DROP + optional ADD proposals for one RQ.
    Returns parsed JSON dict.
    """

    prompt = f"""
You are a research program manager for a PhD project.

Given:
- One existing Research Question (RQ)
- Recent evidence from papers and meeting notes (Japanese text possible)

Task:
Decide ONE of: KEEP, UPDATE, DROP.
Also, if evidence suggests NEW RQs, propose them under new_rq_proposals (can be empty list).

Return STRICT JSON only with this schema:
{{
  "rq_id": "...",
  "decision": "KEEP" | "UPDATE" | "DROP",
  "updated_rq_title": string | null,
  "updated_rq_statement": string | null,
  "reason": string,
  "evidence": {{
    "paper_ids": [string],
    "meeting_ids": [string]
  }},
  "validation_todos": [string],
  "new_rq_proposals": [
    {{
      "title": string,
      "statement": string,
      "reason": string,
      "evidence": {{
        "paper_ids": [string],
        "meeting_ids": [string]
      }},
      "validation_todos": [string]
    }}
  ]
}}

Rules:
- If decision is KEEP, set updated_rq_title and updated_rq_statement to null.
- If decision is DROP, also set updated_* to null and explain why.
- Use evidence IDs that appear in the provided evidence lists.
- Keep the reason concise but specific.

RQ:
- id: {rq.get("rq_id")}
- title: {rq.get("title")}
- rationale: {rq.get("rationale","")}
- proposed_approach: {rq.get("approach","")}
- gap_identified: {rq.get("gap_identified","")}
- tags: {rq.get("tags")}

Relevant Papers (list of dicts):
{json.dumps(relevant.get("papers", []), ensure_ascii=False)}

Relevant Meetings (list of dicts):
{json.dumps(relevant.get("meetings", []), ensure_ascii=False)}
""".strip()

    last_err = None

    for attempt in range(max_retries + 1):
        try:
            res = openai_client.responses.create(
                model=model,
                input=prompt,
            )

            # New SDK: try output_text first
            text = getattr(res, "output_text", None)
            if not text:
                # Fallback: try to reconstruct from output blocks if needed
                try:
                    chunks = []
                    for item in res.output:
                        for c in getattr(item, "content", []) or []:
                            if getattr(c, "type", "") in ("output_text", "text"):
                                chunks.append(getattr(c, "text", ""))
                    text = "\n".join(chunks).strip()
                except Exception:
                    text = None

            data = extract_json_object(text)
            # sanity check
            if data.get("rq_id") is None:
                data["rq_id"] = rq.get("rq_id")

            return data

        except Exception as e:
            last_err = e
            if attempt < max_retries:
                time.sleep(sleep_s * (attempt + 1))
                continue
            raise RuntimeError(f"LLM decision failed after retries. Last error: {last_err}")

rq_decisions = []

for rq in active_rqs:
    rel = select_relevant_evidence_for_rq(rq, evidence_bundle)
    # skip if no evidence at all (optional)
    # if not rel["papers"] and not rel["meetings"]: continue

    decision = llm_rq_decision(rq, rel)
    rq_decisions.append(decision)

print(f"✓ Built rq_decisions: {len(rq_decisions)}")

✓ Built evidence_bundle
  meetings: 1
  papers: 33
✓ Built rq_decisions: 30


In [36]:
# ============================================================
# Cell 09 — Consolidate update bundle (evidence + decisions)
# ============================================================
# Overview:
#   Consolidate evidence (papers + meetings) and LLM decisions into a single
#   update bundle artifact for reproducibility and downstream Notion write-back.
# Inputs / Outputs:
#   Inputs: evidence_bundle (Cell 08), active_rqs (Cell 06), rq_decisions (Cell 09/previous cell)
#   Outputs: update_bundle (dict), saved to UPDATE_BUNDLE_PATH
# Notes:
#   - No dependence on importance or legacy meeting_signals
#   - Keeps Japanese text as-is in meeting notes
#   - Designed for Day24: RQ versioning + draft proposals

import json
from datetime import datetime, timezone
from pathlib import Path

print("\n=== Consolidating update bundle (papers + meetings + rq decisions) ===")

# Safety: ensure required vars exist
if "evidence_bundle" not in globals():
    raise RuntimeError("evidence_bundle not found. Run the evidence bundling cell first.")
if "active_rqs" not in globals():
    raise RuntimeError("active_rqs not found. Run the active RQ loading cell first.")
if "rq_decisions" not in globals():
    raise RuntimeError("rq_decisions not found. Run the LLM decision cell first.")

RUN_TIMESTAMP = globals().get("RUN_TIMESTAMP") or datetime.now(timezone.utc).isoformat()
RECENT_PAPERS_DAYS = globals().get("RECENT_PAPERS_DAYS", None)
RECENT_MEETINGS_DAYS = globals().get("RECENT_MEETINGS_DAYS", None)

# Output path (safe default)
art_dir = Path("./artifacts/day24")
art_dir.mkdir(parents=True, exist_ok=True)
UPDATE_BUNDLE_PATH = globals().get("UPDATE_BUNDLE_PATH") or str(art_dir / f"update_bundle_{RUN_TIMESTAMP[:19].replace(':','-')}.json")

update_bundle = {
    "metadata": {
        "run_timestamp": RUN_TIMESTAMP,
        "time_windows": {
            "papers_days": RECENT_PAPERS_DAYS,
            "meetings_days": RECENT_MEETINGS_DAYS,
        },
        "counts": {
            "papers": len(evidence_bundle.get("papers", [])),
            "meetings": len(evidence_bundle.get("meetings", [])),
            "active_rqs": len(active_rqs),
            "rq_decisions": len(rq_decisions),
        },
    },
    "active_rqs": active_rqs,
    "evidence_bundle": evidence_bundle,
    "rq_decisions": rq_decisions,
}

print("✓ Bundle metadata created")
print(f"  papers: {update_bundle['metadata']['counts']['papers']}")
print(f"  meetings: {update_bundle['metadata']['counts']['meetings']}")
print(f"  active_rqs: {update_bundle['metadata']['counts']['active_rqs']}")
print(f"  rq_decisions: {update_bundle['metadata']['counts']['rq_decisions']}")

# Simple prioritization summary (optional)
decision_counts = {}
for d in rq_decisions:
    decision = (d.get("decision") or "UNKNOWN").upper()
    decision_counts[decision] = decision_counts.get(decision, 0) + 1

update_bundle["summary"] = {
    "decision_counts": decision_counts,
    "add_proposals": sum(len(d.get("new_rq_proposals") or []) for d in rq_decisions),
}

print("✓ Decision summary:")
for k, v in sorted(decision_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  - {k}: {v}")
print(f"  - NEW RQ proposals (total): {update_bundle['summary']['add_proposals']}")

# Save artifact
with open(UPDATE_BUNDLE_PATH, "w", encoding="utf-8") as f:
    json.dump(update_bundle, f, indent=2, ensure_ascii=False)

print(f"✓ Update bundle saved to: {UPDATE_BUNDLE_PATH}")
print("=== Bundle consolidation complete ===")



=== Consolidating update bundle (papers + meetings + rq decisions) ===
✓ Bundle metadata created
  papers: 33
  meetings: 1
  active_rqs: 30
  rq_decisions: 30
✓ Decision summary:
  - KEEP: 18
  - UPDATE: 12
  - NEW RQ proposals (total): 20
✓ Update bundle saved to: ./artifacts/day24/update_bundle_20260120_055814.json
=== Bundle consolidation complete ===


In [40]:
# ============================================================
# Cell 10 — Build RQ proposals from rq_decisions (no extra LLM call)
# ============================================================
# Overview:
#   Convert rq_decisions (KEEP/UPDATE/DROP + ADD proposals) into a unified
#   rq_proposals list suitable for human review and Notion write-back.
# Inputs / Outputs:
#   Inputs: rq_decisions, active_rqs
#   Outputs: rq_proposals (list of dicts)
# Notes:
#   - This replaces the legacy "synthesize_update_signals" pipeline
#   - No dependence on paper_signals/meeting_signals/importance

print("\n=== Building RQ proposals from rq_decisions ===")

if "rq_decisions" not in globals():
    raise RuntimeError("rq_decisions not found. Run the LLM decision cell first.")
if "active_rqs" not in globals():
    raise RuntimeError("active_rqs not found. Run active RQ loading cell first.")

# Map rq_id -> existing rq metadata
rq_map = {rq["rq_id"]: rq for rq in active_rqs}

rq_proposals = []

for d in rq_decisions:
    rq_id = d.get("rq_id")
    decision = (d.get("decision") or "").upper().strip()

    # Common evidence
    ev = d.get("evidence") or {}
    paper_ids = ev.get("paper_ids") or []
    meeting_ids = ev.get("meeting_ids") or []

    base = rq_map.get(rq_id, {})

    if decision == "KEEP":
        # Optional: you can skip KEEP proposals to reduce clutter
        rq_proposals.append({
            "action": "keep",
            "rq_id": rq_id,
            "title": base.get("title") or d.get("updated_rq_title") or "",
            "statement": None,
            "rationale": d.get("reason") or "",
            "evidence": {"paper_ids": paper_ids, "meeting_ids": meeting_ids},
            "validation_todos": d.get("validation_todos") or [],
        })

    elif decision == "UPDATE":
        rq_proposals.append({
            "action": "update",
            "rq_id": rq_id,
            "title": d.get("updated_rq_title") or base.get("title") or "",
            "statement": d.get("updated_rq_statement"),
            "rationale": d.get("reason") or "",
            "evidence": {"paper_ids": paper_ids, "meeting_ids": meeting_ids},
            "validation_todos": d.get("validation_todos") or [],
        })

    elif decision == "DROP":
        rq_proposals.append({
            "action": "drop",
            "rq_id": rq_id,
            "title": base.get("title") or "",
            "statement": None,
            "rationale": d.get("reason") or "",
            "evidence": {"paper_ids": paper_ids, "meeting_ids": meeting_ids},
            "validation_todos": d.get("validation_todos") or [],
        })

    else:
        # Unknown decision: keep as diagnostic
        rq_proposals.append({
            "action": "unknown",
            "rq_id": rq_id,
            "title": base.get("title") or "",
            "statement": None,
            "rationale": f"Unrecognized decision: {decision}. Raw: {d}",
            "evidence": {"paper_ids": paper_ids, "meeting_ids": meeting_ids},
            "validation_todos": d.get("validation_todos") or [],
        })

    # ADD proposals (new RQs)
    for nrq in (d.get("new_rq_proposals") or []):
        nrq_ev = nrq.get("evidence") or {}
        rq_proposals.append({
            "action": "create",
            "rq_id": None,
            "title": nrq.get("title") or "",
            "statement": nrq.get("statement") or "",
            "rationale": nrq.get("reason") or "",
            "evidence": {
                "paper_ids": nrq_ev.get("paper_ids") or [],
                "meeting_ids": nrq_ev.get("meeting_ids") or [],
            },
            "validation_todos": nrq.get("validation_todos") or [],
        })

print(f"✓ Built rq_proposals: {len(rq_proposals)}")

# Optional: if you want to drop KEEP proposals from review, uncomment:
# rq_proposals = [p for p in rq_proposals if p["action"] != "keep"]

# Preview counts
from collections import Counter
counts = Counter([p["action"] for p in rq_proposals])
print("  Proposal counts:", dict(counts))

print("=== RQ proposal build complete ===")

print("\n=== RQ Proposal Overview ===")

for action in ["update", "create", "drop", "keep"]:
    subset = [p for p in rq_proposals if p["action"] == action]
    if not subset:
        continue

    print(f"\n--- {action.upper()} ({len(subset)}) ---")
    for i, p in enumerate(subset[:5], 1):  # 各カテゴリ最大5件
        title = (p.get("title") or "").strip()
        print(f"  {i}. {title[:80]}")
        print(f"     Evidence: papers={len(p['evidence']['paper_ids'])}, meetings={len(p['evidence']['meeting_ids'])}")

def print_proposal_detail(p, max_chars=400):
    print("=" * 60)
    print(f"ACTION: {p['action'].upper()}")
    print(f"TITLE: {p.get('title')}")
    if p.get("statement"):
        print("\nSTATEMENT:")
        print(p["statement"][:max_chars])
    print("\nRATIONALE:")
    print((p.get("rationale") or "")[:max_chars])
    print("\nEVIDENCE:")
    print(f"  paper_ids: {p['evidence']['paper_ids']}")
    print(f"  meeting_ids: {p['evidence']['meeting_ids']}")
    print("\nVALIDATION TODOS:")
    for t in p.get("validation_todos", []):
        print(f"  - {t}")

print("\n=== Detailed Preview: UPDATE proposals ===")
for p in [x for x in rq_proposals if x["action"] == "update"][:3]:
    print_proposal_detail(p)

print("\n=== Detailed Preview: CREATE proposals ===")
for p in [x for x in rq_proposals if x["action"] == "create"][:3]:
    print_proposal_detail(p)

print("\n=== RQ-centric view ===")

from collections import defaultdict
rq_to_props = defaultdict(list)

for p in rq_proposals:
    rq_to_props[p.get("rq_id")].append(p)

for rq_id, props in list(rq_to_props.items())[:5]:
    print("\n" + "-" * 60)
    print(f"RQ ID: {rq_id}")
    for p in props:
        print(f"  - {p['action'].upper()}: {p.get('title')[:60]}")



=== Building RQ proposals from rq_decisions ===
✓ Built rq_proposals: 50
  Proposal counts: {'keep': 18, 'create': 20, 'update': 12}
=== RQ proposal build complete ===

=== RQ Proposal Overview ===

--- UPDATE (12) ---
  1. 日本のスタートアップの独自成長モデルの制度設計における資金調達手法と組織内起業（イントレプレナーシップ）の役割の解明
     Evidence: papers=2, meetings=0
  2. 日本のディープテックスタートアップにおける資金不足問題は、どのような政策設計や投資家（特にVC/CVC）の戦略的関与によって解消可能か？－投資家タイプ別のエグジ
     Evidence: papers=3, meetings=1
  3. 大学のデュアルユース研究制約をオフキャンパス研究推進と企業ベンチャーキャピタル戦略で緩和する政策は、イノベーション創出と安全保障の両立を可能にするか？
     Evidence: papers=1, meetings=1
  4. クロスボーダー展開スタートアップにおける企業連携が現地市場適応と資金調達双方に与える相乗効果の条件とその多様化要因は何か？
     Evidence: papers=5, meetings=1
  5. 企業とスタートアップの協業における戦略的投資と財務投資のトレードオフを克服する組織設計および契約条件の要因は何か？
     Evidence: papers=3, meetings=1

--- CREATE (20) ---
  1. 多様な投資主体が起業家の成功に及ぼす制度的及びネットワーク支援の比較分析
     Evidence: papers=1, meetings=1
  2. 日本のディープテックスタートアップ資金調達における地理的・ジェンダーバイアスはどの程度存在し、どのような政策で除去可能か？
     Evidence: papers=1, meetings=0
  3. ベンチャーキャピタルのネットワーク中心性がディープテックスタート

In [43]:
# ============================================================
# Cell 11 — Prepare Notion payloads (DB-aligned, no version property)
# ============================================================
# Overview:
#   Prepare Notion create-page payloads for RQ proposals using the actual
#   RQ DB schema (Name/Status/Priority/Tags/Linked Paper/Rationale/Approach/Gap).
#   Since the DB has no explicit version property or parent relation, versioning is:
#     - encoded in the title prefix, e.g., "[v1.1] ..."
#     - and parent RQ reference is written into Rationale/Background.
# Inputs / Outputs:
#   Inputs: rq_proposals (Cell 10), active_rqs (Cell 06), RQ_SCHEMA (aligned)
#   Outputs: notion_payloads (list of dicts ready for Notion API)
# Notes:
#   - No Notion writes here; just prepares payloads
#   - Draft status is used for all created pages
#   - UPDATE is represented as a new Draft page (no overwrite)

print("\n=== Preparing Notion payloads (DB-aligned) ===")

# --- Ensure RQ_SCHEMA matches your DIAG ---
# Expected keys: title/status/priority/tags/evidence/rationale/approach/gap
# If you used different names, update here accordingly.
RQ_SCHEMA = {
    "title": "Name",
    "status": "Status",
    "priority": "Priority",
    "tags": "Tags",
    "evidence": "Linked Paper",
    "rationale": "Rationale / Background",
    "approach": "Proposed Approach",
    "gap": "Gap Identified",
}

# Versioning rules (title prefix only)
VERSIONING_RULES = {"minor": 0.1, "major": 1.0}
DEFAULT_VERSION_INCREMENT = "minor"

# Build lookup (no version in DB, so default base version = 1.0)
rq_lookup = {}
for rq in active_rqs:
    rq_lookup[rq["rq_id"]] = {
        "title": rq.get("title", ""),
        "priority": rq.get("priority", None),
        "tags": rq.get("tags", []),
        "evidence_ids": rq.get("evidence_ids", []),
        "base_version": rq.get("base_version", 1.0),  # you can later store this if you add a version prop
    }

print(f"✓ Built lookup for {len(rq_lookup)} active RQs")

def make_versioned_title(base_title: str, old_v: float, inc_type: str) -> str:
    inc = VERSIONING_RULES.get(inc_type, VERSIONING_RULES["minor"])
    new_v = round(float(old_v) + float(inc), 1)
    prefix = f"[v{new_v}]"
    # avoid double prefixing
    if base_title.strip().startswith("[v"):
        return base_title
    return f"{prefix} {base_title}".strip(), new_v

def notion_rich_text(content: str):
    return [{"text": {"content": content}}]

def safe_priority_name(p: dict, fallback: str = "Medium") -> str:
    # proposal may not have priority; keep stable defaults
    val = (p.get("priority") or "").strip()
    return val if val else fallback

notion_payloads = []

for i, p in enumerate(rq_proposals, 1):
    action = p.get("action")
    if action not in ("update", "create", "drop"):
        # usually ignore keep to reduce clutter
        continue

    payload = {
        "proposal_index": i,
        "action": action,
        "original_proposal": p,
        "notion_create": {
            "parent": {"database_id": RQ_DB_ID},
            "properties": {},
            "children": []
        },
        "metadata": {}
    }

    # Evidence ids from proposal shape
    ev = p.get("evidence") or {}
    evidence_ids = list(set((ev.get("paper_ids") or [])))  # papers are Notion page IDs if relation points to Literature DB

    # --- CREATE: new RQ draft ---
    if action == "create":
        title = (p.get("title") or "New RQ").strip()
        statement = (p.get("statement") or "").strip()
        rationale = (p.get("rationale") or "").strip()

        payload["notion_create"]["properties"] = {
            RQ_SCHEMA["title"]: {"title": [{"text": {"content": title[:200]}}]},
            RQ_SCHEMA["status"]: {"status": {"name": "Draft"}},
            RQ_SCHEMA["priority"]: {"select": {"name": safe_priority_name(p)}},
            RQ_SCHEMA["tags"]: {"multi_select": [{"name": t} for t in (p.get("tags") or [])][:50]},
            RQ_SCHEMA["rationale"]: {"rich_text": notion_rich_text((rationale or "Justification TBD")[:2000])},
        }

        # Put statement/approach/gap into optional fields if you want
        if statement:
            payload["notion_create"]["properties"][RQ_SCHEMA["approach"]] = {
                "rich_text": notion_rich_text(statement[:2000])
            }

        # Evidence relation
        if evidence_ids:
            payload["notion_create"]["properties"][RQ_SCHEMA["evidence"]] = {
                "relation": [{"id": eid} for eid in evidence_ids[:25]]
            }

        # Children blocks (optional)
        todos = p.get("validation_todos") or []
        if todos:
            payload["notion_create"]["children"] = (
                [
                    {
                        "object": "block",
                        "type": "heading_2",
                        "heading_2": {
                            "rich_text": [{"text": {"content": "Validation ToDos"}}]
                        },
                    }
                ]
                +
                [
                    {
                        "object": "block",
                        "type": "bulleted_list_item",
                        "bulleted_list_item": {
                            "rich_text": [{"text": {"content": t[:200]}}]
                        },
                    }
                    for t in todos[:20]
                ]
            )


        payload["metadata"] = {"evidence_count": len(evidence_ids)}
        notion_payloads.append(payload)
        continue

    # --- UPDATE: new draft page referencing parent RQ ---
    if action == "update":
        parent_id = p.get("rq_id")
        parent_info = rq_lookup.get(parent_id)
        if not parent_info:
            print(f"  ⚠ Proposal {i}: parent RQ not found, skipping")
            continue

        base_title = (p.get("title") or parent_info["title"] or "RQ").strip()
        old_v = float(parent_info.get("base_version", 1.0))
        inc_type = p.get("version_increment") or DEFAULT_VERSION_INCREMENT

        versioned_title, new_v = make_versioned_title(base_title, old_v, inc_type)

        rationale = (p.get("rationale") or "").strip()
        parent_ref = f"Parent RQ: {parent_info['title']} (id={parent_id})"
        combined_rationale = (parent_ref + "\n\n" + rationale).strip()

        statement = (p.get("statement") or "").strip()

        # Merge tags: keep parent tags + proposal tags
        merged_tags = []
        seen = set()
        for t in (parent_info.get("tags") or []) + (p.get("tags") or []):
            if not t:
                continue
            if t not in seen:
                seen.add(t)
                merged_tags.append(t)

        payload["notion_create"]["properties"] = {
            RQ_SCHEMA["title"]: {"title": [{"text": {"content": versioned_title[:200]}}]},
            RQ_SCHEMA["status"]: {"status": {"name": "Draft"}},
            RQ_SCHEMA["priority"]: {"select": {"name": safe_priority_name(p, fallback=(parent_info.get("priority") or "Medium"))}},
            RQ_SCHEMA["tags"]: {"multi_select": [{"name": t} for t in merged_tags][:50]},
            RQ_SCHEMA["rationale"]: {"rich_text": notion_rich_text(combined_rationale[:2000])},
        }

        if statement:
            payload["notion_create"]["properties"][RQ_SCHEMA["approach"]] = {
                "rich_text": notion_rich_text(statement[:2000])
            }

        # Evidence = parent evidence + proposal evidence (union)
        merged_evidence = list(set((parent_info.get("evidence_ids") or []) + evidence_ids))
        if merged_evidence:
            payload["notion_create"]["properties"][RQ_SCHEMA["evidence"]] = {
                "relation": [{"id": eid} for eid in merged_evidence[:25]]
            }

        todos = p.get("validation_todos") or []
        if todos:
            payload["notion_create"]["children"] = [
                {"object": "block", "type": "heading_2",
                 "heading_2": {"rich_text": [{"text": {"content": "Validation ToDos"}}]}},
            ] + [
                {"object": "block", "type": "bulleted_list_item",
                 "bulleted_list_item": {"rich_text": [{"text": {"content": t[:200]}}]}}
                for t in todos[:20]
            ]

        payload["metadata"] = {
            "parent_rq_id": parent_id,
            "old_version_assumed": old_v,
            "new_version_assumed": new_v,
            "version_increment": inc_type,
            "evidence_count": len(merged_evidence),
        }

        notion_payloads.append(payload)
        continue

    # --- DROP: represent as a draft note (or optionally update existing status) ---
    if action == "drop":
        parent_id = p.get("rq_id")
        parent_info = rq_lookup.get(parent_id, {})
        title = f"[DROP CANDIDATE] {parent_info.get('title','')}".strip()
        rationale = (p.get("rationale") or "No rationale provided").strip()

        payload["notion_create"]["properties"] = {
            RQ_SCHEMA["title"]: {"title": [{"text": {"content": title[:200]}}]},
            RQ_SCHEMA["status"]: {"status": {"name": "Draft"}},
            RQ_SCHEMA["priority"]: {"select": {"name": "Low"}},
            RQ_SCHEMA["tags"]: {"multi_select": [{"name": "drop_candidate"}]},
            RQ_SCHEMA["rationale"]: {"rich_text": notion_rich_text(rationale[:2000])},
        }

        if evidence_ids:
            payload["notion_create"]["properties"][RQ_SCHEMA["evidence"]] = {
                "relation": [{"id": eid} for eid in evidence_ids[:25]]
            }

        payload["metadata"] = {"parent_rq_id": parent_id, "evidence_count": len(evidence_ids)}
        notion_payloads.append(payload)

print(f"✓ Prepared Notion payloads: {len(notion_payloads)}")

# Quick preview
from collections import Counter
c = Counter([p["action"] for p in notion_payloads])
print("  Payload counts:", dict(c))
if notion_payloads:
    print("\n  Sample payload titles:")
    for x in notion_payloads[:5]:
        t_prop = x["notion_create"]["properties"][RQ_SCHEMA["title"]]["title"][0]["text"]["content"]
        print(f"   - ({x['action']}) {t_prop}")



=== Preparing Notion payloads (DB-aligned) ===
✓ Built lookup for 30 active RQs
✓ Prepared Notion payloads: 32
  Payload counts: {'create': 20, 'update': 12}

  Sample payload titles:
   - (create) 多様な投資主体が起業家の成功に及ぼす制度的及びネットワーク支援の比較分析
   - (update) [v1.1] 日本のスタートアップの独自成長モデルの制度設計における資金調達手法と組織内起業（イントレプレナーシップ）の役割の解明
   - (update) [v1.1] 日本のディープテックスタートアップにおける資金不足問題は、どのような政策設計や投資家（特にVC/CVC）の戦略的関与によって解消可能か？－投資家タイプ別のエグジット戦略とネットワーク形成の影響を踏まえて
   - (create) 日本のディープテックスタートアップ資金調達における地理的・ジェンダーバイアスはどの程度存在し、どのような政策で除去可能か？
   - (create) ベンチャーキャピタルのネットワーク中心性がディープテックスタートアップの資金調達成功に与える影響


In [45]:
# ============================================================
# Cell 12 — Human review helpers (preview table, dry-run mode)
# ============================================================
# Overview:
#   Provides human-readable preview of RQ proposals before Notion write-back.
#   Generates formatted tables, comparison views, and dry-run summaries.
#   Enables informed decision-making about which proposals to accept.
# Inputs / Outputs:
#   Inputs: rq_proposals (from Cell 10), notion_payloads (from Cell 11), active_rqs
#   Outputs: Preview tables (console), dry-run report, filtered proposals
# Notes:
#   - Uses pandas for clean tabular display
#   - Highlights high-priority and high-confidence proposals
#   - Shows before/after comparison for updates
#   - Dry-run mode safety check before Cell 13 execution

print("\n=== Human Review: RQ Proposals Preview ===")

import pandas as pd
from collections import Counter

if not rq_proposals:
    print("✗ No proposals to preview. Run the proposal build cell first.")
else:
    preview_data = []

    for i, proposal in enumerate(rq_proposals, 1):
        ev = proposal.get("evidence") or {}
        paper_n = len(ev.get("paper_ids") or [])
        meeting_n = len(ev.get("meeting_ids") or [])

        row = {
            "#": i,
            "Action": (proposal.get("action") or "").upper(),
            "Title": ((proposal.get("title") or "")[:50] + "...") if len(proposal.get("title") or "") > 50 else (proposal.get("title") or ""),
            "Evidence(papers)": paper_n,
            "Evidence(meetings)": meeting_n,
            "Rationale": ((proposal.get("rationale") or "")[:60] + "...") if len(proposal.get("rationale") or "") > 60 else (proposal.get("rationale") or ""),
        }

        # Version info: derive from notion_payloads metadata if available
        if proposal.get("action") == "update" and "notion_payloads" in globals() and notion_payloads:
            matching_payload = next(
                (p for p in notion_payloads if p.get("original_proposal") == proposal),
                None
            )
            if matching_payload and matching_payload.get("metadata"):
                meta = matching_payload["metadata"]
                row["Version"] = f"{meta.get('old_version_assumed','?')} → {meta.get('new_version_assumed','?')}"
            else:
                row["Version"] = "N/A"
        elif proposal.get("action") == "create":
            row["Version"] = "v1.0 (new)"
        else:
            row["Version"] = ""

        preview_data.append(row)

    preview_df = pd.DataFrame(preview_data)

    print(f"\n✓ Generated preview for {len(preview_df)} proposals\n")
    print(preview_df.to_string(index=False))

    # Action breakdown
    print("\n--- Action Breakdown ---")
    action_counts = Counter(preview_df["Action"])
    for k, v in sorted(action_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {k}: {v}")

    # Quick highlight: updates with evidence
    updates_with_evidence = preview_df[
        (preview_df["Action"] == "UPDATE") &
        ((preview_df["Evidence(papers)"] > 0) | (preview_df["Evidence(meetings)"] > 0))
    ]
    if not updates_with_evidence.empty:
        print("\n--- UPDATE proposals with evidence (top 10) ---")
        print(updates_with_evidence[["#", "Title", "Version", "Evidence(papers)", "Evidence(meetings)"]].head(10).to_string(index=False))

    # Before/After comparison for updates (title-level)
    update_proposals = [p for p in rq_proposals if p.get("action") == "update"]
    if update_proposals:
        print("\n--- Update Comparison (Before → After) ---")
        for j, proposal in enumerate(update_proposals[:5], 1):
            parent_id = proposal.get("rq_id")
            parent_rq = next((rq for rq in active_rqs if rq.get("rq_id") == parent_id), None)
            print(f"\n  Update #{j}:")
            if parent_rq:
                print(f"    BEFORE: {parent_rq.get('title','')[:90]}")
            else:
                print(f"    BEFORE: (parent rq not found: {parent_id})")
            print(f"    AFTER:  {proposal.get('title','')[:90]}")
            print(f"    WHY:    {(proposal.get('rationale') or '')[:140]}...")

    print("\n=== Human review preview complete ===")
    print("Next: If payloads are prepared, proceed to Notion write-back cell (dry-run recommended).")




=== Human Review: RQ Proposals Preview ===

✓ Generated preview for 50 proposals

 # Action                                                 Title  Evidence(papers)  Evidence(meetings)                                                       Rationale    Version
 1   KEEP    CVCの成長過程における学習効果は、ファイナンシャルリターン重視の姿勢によってどのように変化するか？                 1                   1 最新の研究はCVCの財務リターン重視が価値創出効果に限定的であり、戦略的目的が重要であることを示しているが、ファイナンシ...           
 2   KEEP 日本と米国の上場基準やExit環境の制度差が、政府系VC（GVC）を活用するスタートアップのExit...                 0                   1 No direct new evidence updating or challenging the original ...           
 3   KEEP       政府系VCがファイナンシャルリターンを重視する起業家の成功に与える影響とそのメカニズムは何か？                 3                   1 現在の証拠は政府系VCのファイナンシャルリターン志向の起業家に対する影響の直接検証や制度環境・ネットワーク資源のメカニズ...           
 4 CREATE                  多様な投資主体が起業家の成功に及ぼす制度的及びネットワーク支援の比較分析                 1                   1 代替投資ファンドの存在やCVCの投資戦略に関する最新議論が示唆する多様な資金提供者の役割把握が、政府系VCの効果をより精... v1.0 (new)
 5 UPDATE 日本のスタートアップの独自成長モデルの制度設

In [58]:
# ============================================================
# Cell 13 — Write proposals to Notion (with dry_run switch) [FIXED]
# ============================================================
# Overview:
#   Executes Notion write-back for approved RQ proposals.
#   Respects DRY_RUN flag. Creates new pages in RQ DB with:
#     - Correct Status option (NOT "Draft"; uses "New"/fallback)
#     - Evidence relations (if provided by Cell 11)
#     - Optional children blocks
#   Includes robust error handling + summary.
#
# Inputs / Outputs:
#   Inputs: notion_payloads (from Cell 11), DRY_RUN (from Cell 02)
#   Outputs: write_results (list), console summary
#
# Notes:
#   - Supports BOTH payload formats:
#       (A) legacy: payload['notion_properties'], payload.get('notion_children')
#       (B) new:    payload['notion_create'] = {'properties':..., 'children':...}
#   - Automatically fixes invalid Status names by selecting an allowed option
#     from your RQ DB: ["Not started", "New", "Under Review", "In progress", "Done"].
#   - If a relation error occurs (page not shared), it logs the error; does not crash.
#   - Requires: NOTION_API_BASE, NOTION_HEADERS, requests, create_page, RQ_SCHEMA, RQ_DB_ID

import time
import requests
from typing import Dict, Any, Tuple, List, Optional

print("\n=== Writing RQ Proposals to Notion ===")
print(f"DRY_RUN mode: {DRY_RUN}")
print(f"Target database: {RQ_DB_ID[:8]}...")
print(f"Payloads to process: {len(notion_payloads)}")

# -----------------------------
# Helpers
# -----------------------------
def notion_get(path: str, timeout: int = 30) -> dict:
    url = f"{NOTION_API_BASE}{path}"
    r = requests.get(url, headers=NOTION_HEADERS, timeout=timeout)
    if r.status_code >= 300:
        raise RuntimeError(f"Notion GET {path} failed: {r.status_code} {r.text}")
    return r.json()

def get_database_status_options(database_id: str, status_prop_name: str) -> List[str]:
    db = notion_get(f"/databases/{database_id}")
    prop = db["properties"][status_prop_name]
    if prop["type"] != "status":
        raise ValueError(f"Property '{status_prop_name}' is not type=status (found {prop['type']})")
    return [o["name"] for o in prop["status"]["options"]]

def choose_default_status(options: List[str]) -> str:
    # Prefer "New" as the closest to "Draft"
    preferred = ["New", "Not started", "Under Review", "In progress", "Done"]
    for cand in preferred:
        if cand in options:
            return cand
    return options[0] if options else "New"

def get_payload_parts(payload: Dict[str, Any]) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    """
    Supports both payload formats:
      new:    payload['notion_create']['properties'], payload['notion_create'].get('children', [])
      legacy: payload['notion_properties'], payload.get('notion_children', [])
    """
    if "notion_create" in payload:
        props = payload["notion_create"]["properties"]
        children = payload["notion_create"].get("children", []) or []
        return props, children
    else:
        props = payload["notion_properties"]
        children = payload.get("notion_children", []) or []
        return props, children

def safe_title_from_properties(properties: Dict[str, Any], title_prop: str) -> str:
    try:
        return properties[title_prop]["title"][0]["text"]["content"]
    except Exception:
        return "(untitled)"

# -----------------------------
# Resolve status option
# -----------------------------
STATUS_PROP_NAME = RQ_SCHEMA["status"]  # likely "Status"
try:
    status_options = get_database_status_options(RQ_DB_ID, STATUS_PROP_NAME)
    DEFAULT_STATUS = choose_default_status(status_options)
    print(f"✅ Status options detected: {status_options}")
    print(f"✅ DEFAULT_STATUS selected: {DEFAULT_STATUS}")
except Exception as e:
    # If we cannot introspect (rare), fallback to "New"
    status_options = ["Not started", "New", "Under Review", "In progress", "Done"]
    DEFAULT_STATUS = "New"
    print(f"⚠ Could not introspect DB status options; fallback DEFAULT_STATUS='{DEFAULT_STATUS}'. Error: {e}")

# -----------------------------
# Main
# -----------------------------
if not notion_payloads:
    print("✗ No payloads to write. Run Cell 11 to prepare payloads.")
    write_results = []
    success_count = 0
    failure_count = 0
else:
    write_results = []
    success_count = 0
    failure_count = 0

    # --- DRY RUN MODE ---
    if DRY_RUN:
        print("\n🔍 DRY RUN MODE: Simulating writes (no actual API calls)\n")

        for payload in notion_payloads:
            proposal_idx = payload.get("proposal_index")
            action = payload.get("action")
            try:
                properties, children = get_payload_parts(payload)

                # Ensure valid status option
                properties[STATUS_PROP_NAME] = {"status": {"name": DEFAULT_STATUS}}

                title = safe_title_from_properties(properties, RQ_SCHEMA["title"])
                evidence_count = payload.get("metadata", {}).get("evidence_count", 0)

                # Simulate success
                result = {
                    "proposal_index": proposal_idx,
                    "action": action,
                    "title": title,
                    "status": "success (dry-run)",
                    "page_id": f"dry_run_page_{proposal_idx}",
                    "page_url": None,
                    "error": None,
                    "metadata": payload.get("metadata", {}),
                }
                write_results.append(result)
                success_count += 1

                print(f"  [{proposal_idx}] {str(action).upper()} (dry-run): {title[:70]}...")
                print(f"      Status: {DEFAULT_STATUS}, Evidence: {evidence_count}, Children blocks: {len(children)}\n")

            except Exception as e:
                failure_count += 1
                err = str(e)
                print(f"  ✗ Dry-run simulation error for payload {proposal_idx}: {err}\n")
                write_results.append({
                    "proposal_index": proposal_idx,
                    "action": action,
                    "title": "N/A",
                    "status": "error (dry-run)",
                    "page_id": None,
                    "page_url": None,
                    "error": err,
                    "metadata": {},
                })

        print(f"✓ Dry-run complete: {success_count} simulated writes, {failure_count} errors")
        print("  Set DRY_RUN=False in Cell 02 to execute actual writes.")

    # --- LIVE WRITE MODE ---
    else:
        print("\n⚠ LIVE WRITE MODE: Executing actual Notion API writes\n")
        print("Writing proposals to Notion RQ database...\n")

        for payload in notion_payloads:
            proposal_idx = payload.get("proposal_index")
            action = payload.get("action")

            try:
                properties, children = get_payload_parts(payload)

                # Ensure valid status option
                properties[STATUS_PROP_NAME] = {"status": {"name": DEFAULT_STATUS}}

                title = safe_title_from_properties(properties, RQ_SCHEMA["title"])
                print(f"  [{proposal_idx}] {str(action).upper()}: {title[:70]}...")

                created_page = create_page(
                    database_id=RQ_DB_ID,
                    properties=properties,
                    children=children if children else None
                )

                if not created_page:
                    raise RuntimeError("create_page returned None")

                page_id = created_page.get("id")
                page_url = created_page.get("url")

                write_results.append({
                    "proposal_index": proposal_idx,
                    "action": action,
                    "title": title,
                    "status": "success",
                    "page_id": page_id,
                    "page_url": page_url,
                    "error": None,
                    "metadata": payload.get("metadata", {}),
                })
                success_count += 1

                print(f"      ✓ Created: {str(page_id)[:20]}...")
                if page_url:
                    print(f"      URL: {page_url}")
                print("")

                # light rate limit
                time.sleep(0.2)

            except Exception as e:
                failure_count += 1
                err = str(e)

                # Optional hinting: relation visibility is the most common second issue
                hint = ""
                if "Could not find page with ID" in err or "Make sure the relevant pages and databases are shared" in err:
                    hint = " (Hint: share the parent/related pages + DB with your Notion integration.)"
                print(f"      ✗ Write failed: {err}{hint}\n")

                write_results.append({
                    "proposal_index": proposal_idx,
                    "action": action,
                    "title": title if "title" in locals() else "N/A",
                    "status": "error",
                    "page_id": None,
                    "page_url": None,
                    "error": err,
                    "metadata": payload.get("metadata", {}),
                })

        print(f"\n{'='*60}")
        print(f"Write complete: {success_count} succeeded, {failure_count} failed")

        if failure_count > 0:
            print("\n⚠ Failed writes (review and retry):")
            for r in write_results:
                if r.get("status") == "error":
                    t = (r.get("title") or "N/A")[:50]
                    print(f"  - [{r.get('proposal_index')}] {t}...")
                    print(f"    Error: {(r.get('error') or '')[:120]}...")

# --- Summary Statistics ---
print("\n=== Write Results Summary ===")
print(f"Total proposals: {len(notion_payloads)}")
print(f"Successful writes: {success_count}")
print(f"Failed writes: {failure_count}")
print(f"Success rate: {(success_count / len(notion_payloads) * 100) if notion_payloads else 0:.1f}%")

if write_results:
    updates = [r for r in write_results if r.get("action") == "update" and "success" in (r.get("status") or "")]
    creates = [r for r in write_results if r.get("action") == "create" and "success" in (r.get("status") or "")]
    print(f"\n  Updates written: {len(updates)}")
    print(f"  New RQs created: {len(creates)}")

    successes = [r for r in write_results if "success" in (r.get("status") or "")]
    if successes:
        print("\n  Sample successful writes:")
        for r in successes[:3]:
            print(f"    [{r.get('proposal_index')}] {str(r.get('action')).upper()}: {(r.get('title') or '')[:60]}...")
            if not DRY_RUN and r.get("page_url"):
                print(f"        {r.get('page_url')}")

print("\n=== Notion write-back complete ===")
print("Next step: Run Cell 14 to save output artifacts")
print(f"\n✓ Write results stored ({len(write_results)} records) for artifact export")



=== Writing RQ Proposals to Notion ===
DRY_RUN mode: False
Target database: 2a98e0e4...
Payloads to process: 32
✅ Status options detected: ['Not started', 'New', 'Under Review', 'In progress', 'Done']
✅ DEFAULT_STATUS selected: New

⚠ LIVE WRITE MODE: Executing actual Notion API writes

Writing proposals to Notion RQ database...

  [4] CREATE: 多様な投資主体が起業家の成功に及ぼす制度的及びネットワーク支援の比較分析...
      ✓ Created: 2ee8e0e4-d162-8119-b...
      URL: https://www.notion.so/2ee8e0e4d1628119b241c163f12d4a42

  [5] UPDATE: [v1.1] 日本のスタートアップの独自成長モデルの制度設計における資金調達手法と組織内起業（イントレプレナーシップ）の役割の解明...
      ✓ Created: 2ee8e0e4-d162-81db-a...
      URL: https://www.notion.so/v1-1-2ee8e0e4d16281dba984c1409f4cd332

  [7] UPDATE: [v1.1] 日本のディープテックスタートアップにおける資金不足問題は、どのような政策設計や投資家（特にVC/CVC）の戦略的関与によって解...
      ✓ Created: 2ee8e0e4-d162-81a8-9...
      URL: https://www.notion.so/v1-1-VC-CVC-2ee8e0e4d16281a899e7e45e87929aa0

  [8] CREATE: 日本のディープテックスタートアップ資金調達における地理的・ジェンダーバイアスはどの程度存在し、どのような政策で除去可能か？...
      ✓ Created: 2ee8e

In [59]:
# ============================================================
# Cell 14 — Save output artifacts (JSON and CSV)
# ============================================================
# Overview:
#   Export all generated data to persistent artifacts for reproducibility,
#   auditing, and external review. Saves proposals, write results, and
#   synthesis data in both JSON (structured) and CSV (tabular) formats.
# Inputs / Outputs:
#   Inputs: rq_proposals (Cell 10), write_results (Cell 13), synthesis_result (Cell 10),
#           update_bundle (Cell 09), ARTIFACTS_DIR, RUN_TIMESTAMP
#   Outputs: JSON files (proposals, results, bundle), CSV files (proposals, results)
# Notes:
#   - All artifacts timestamped with RUN_TIMESTAMP
#   - JSON preserves full structure; CSV flattens for spreadsheet review
#   - Handles empty datasets gracefully (no error if no proposals)
#   - File paths echo to console for easy access

print("\n=== Saving output artifacts ===")
print(f"Artifact directory: {ARTIFACTS_DIR}")
print(f"Run timestamp: {RUN_TIMESTAMP}")

# --- Artifact 1: RQ Proposals (JSON) ---
print("\n1. Saving RQ proposals (JSON)...")

try:
    proposals_export = {
        'metadata': {
            'run_timestamp': RUN_TIMESTAMP,
            'total_proposals': len(rq_proposals),
            'dry_run_mode': DRY_RUN,
            'openai_model': OPENAI_CONFIG['model'],
            'generation_params': {
                'temperature': OPENAI_CONFIG['temperature'],
                'max_tokens': OPENAI_CONFIG['max_tokens']
            }
        },
        'proposals': rq_proposals
    }
    
    with open(PROPOSALS_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(proposals_export, f, indent=2, ensure_ascii=False)
    
    print(f"   ✓ Saved {len(rq_proposals)} proposals to: {PROPOSALS_JSON_PATH}")
    
except Exception as e:
    print(f"   ✗ Error saving proposals JSON: {e}")

# --- Artifact 2: RQ Proposals (CSV) ---
print("\n2. Saving RQ proposals (CSV)...")

try:
    if rq_proposals:
        # Flatten proposals for CSV
        csv_rows = []
        for i, proposal in enumerate(rq_proposals, 1):
            row = {
                'proposal_id': i,
                'action': proposal['action'],
                'rq_id': proposal.get('rq_id', ''),
                'title': proposal['title'],
                'description': proposal['description'][:500] + '...' if len(proposal['description']) > 500 else proposal['description'],
                'rationale': proposal['rationale'][:500] + '...' if len(proposal['rationale']) > 500 else proposal['rationale'],
                'priority': proposal['priority'],
                'confidence': proposal['confidence'],
                'evidence_count': len(proposal.get('evidence_ids', [])),
                'evidence_ids': '|'.join(proposal.get('evidence_ids', [])[:10]),  # First 10
                'version_increment': proposal.get('version_increment', 'N/A'),
                'tags': '|'.join(proposal.get('tags', []))
            }
            csv_rows.append(row)
        
        proposals_df = pd.DataFrame(csv_rows)
        proposals_df.to_csv(PROPOSALS_CSV_PATH, index=False)
        
        print(f"   ✓ Saved {len(csv_rows)} proposals to: {PROPOSALS_CSV_PATH}")
    else:
        print("   ⚠ No proposals to save (empty dataset)")
        # Create empty CSV with headers
        pd.DataFrame(columns=['proposal_id', 'action', 'title', 'priority', 'confidence']).to_csv(PROPOSALS_CSV_PATH, index=False)
        print(f"   ✓ Created empty CSV: {PROPOSALS_CSV_PATH}")
        
except Exception as e:
    print(f"   ✗ Error saving proposals CSV: {e}")

# --- Artifact 3: Write Results (JSON) ---
print("\n3. Saving write results (JSON)...")

write_results_path = os.path.join(ARTIFACTS_DIR, f'write_results_{RUN_TIMESTAMP}.json')

try:
    write_results_export = {
        'metadata': {
            'run_timestamp': RUN_TIMESTAMP,
            'dry_run_mode': DRY_RUN,
            'total_attempts': len(write_results),
            'success_count': sum(1 for r in write_results if 'success' in r.get('status', '')),
            'failure_count': sum(1 for r in write_results if r.get('status') == 'error'),
            'target_database_id': RQ_DB_ID
        },
        'results': write_results
    }
    
    with open(write_results_path, 'w', encoding='utf-8') as f:
        json.dump(write_results_export, f, indent=2, ensure_ascii=False)
    
    print(f"   ✓ Saved {len(write_results)} write results to: {write_results_path}")
    
except Exception as e:
    print(f"   ✗ Error saving write results JSON: {e}")

# --- Artifact 4: Write Results (CSV) ---
print("\n4. Saving write results (CSV)...")

write_results_csv_path = os.path.join(ARTIFACTS_DIR, f'write_results_{RUN_TIMESTAMP}.csv')

try:
    if write_results:
        # Flatten write results for CSV
        csv_rows = []
        for result in write_results:
            row = {
                'proposal_index': result.get('proposal_index'),
                'action': result.get('action'),
                'title': result.get('title', '')[:200],
                'status': result.get('status'),
                'page_id': result.get('page_id', ''),
                'page_url': result.get('page_url', ''),
                'error': result.get('error', '')[:500] if result.get('error') else '',
                'old_version': result.get('metadata', {}).get('old_version', ''),
                'new_version': result.get('metadata', {}).get('new_version', ''),
                'evidence_count': result.get('metadata', {}).get('evidence_count', 0)
            }
            csv_rows.append(row)
        
        results_df = pd.DataFrame(csv_rows)
        results_df.to_csv(write_results_csv_path, index=False)
        
        print(f"   ✓ Saved {len(csv_rows)} write results to: {write_results_csv_path}")
    else:
        print("   ⚠ No write results to save (empty dataset)")
        # Create empty CSV
        pd.DataFrame(columns=['proposal_index', 'action', 'title', 'status']).to_csv(write_results_csv_path, index=False)
        print(f"   ✓ Created empty CSV: {write_results_csv_path}")
        
except Exception as e:
    print(f"   ✗ Error saving write results CSV: {e}")

# --- Artifact 5: Decision results (JSON) ---
print("\n5. Saving decision results (JSON)...")

decisions_path = os.path.join(ARTIFACTS_DIR, f'rq_decisions_{RUN_TIMESTAMP}.json')

try:
    if 'rq_decisions' in globals() and rq_decisions:
        export = {
            "metadata": {
                "run_timestamp": RUN_TIMESTAMP,
                "openai_model": OPENAI_CONFIG.get("model", "unknown") if "OPENAI_CONFIG" in globals() else "unknown",
            },
            "rq_decisions": rq_decisions
        }
        with open(decisions_path, 'w', encoding='utf-8') as f:
            json.dump(export, f, indent=2, ensure_ascii=False)
        print(f"   ✓ Saved rq_decisions to: {decisions_path}")
    else:
        print("   ⚠ No rq_decisions available (skipped)")
except Exception as e:
    print(f"   ✗ Error saving rq_decisions JSON: {e}")

# --- Artifact 6: Evidence bundle (JSON) ---
print("\n6. Saving evidence bundle (JSON)...")

evidence_path = os.path.join(ARTIFACTS_DIR, f'evidence_bundle_{RUN_TIMESTAMP}.json')

try:
    if 'evidence_bundle' in globals() and evidence_bundle:
        export = {
            "metadata": {"run_timestamp": RUN_TIMESTAMP},
            "evidence_bundle": evidence_bundle
        }
        with open(evidence_path, 'w', encoding='utf-8') as f:
            json.dump(export, f, indent=2, ensure_ascii=False)
        print(f"   ✓ Saved evidence_bundle to: {evidence_path}")
    else:
        print("   ⚠ No evidence_bundle available (skipped)")
except Exception as e:
    print(f"   ✗ Error saving evidence_bundle JSON: {e}")

# --- Artifact 7: Complete run manifest ---
print("\n7. Saving run manifest...")

manifest_path = os.path.join(ARTIFACTS_DIR, f'run_manifest_{RUN_TIMESTAMP}.json')

try:
    manifest = {
        'run_timestamp': RUN_TIMESTAMP,
        'configuration': {
            'dry_run': DRY_RUN,
            'openai_model': OPENAI_CONFIG['model'],
            'openai_temperature': OPENAI_CONFIG['temperature'],
            'recent_papers_days': RECENT_PAPERS_DAYS,
            'recent_meetings_days': RECENT_MEETINGS_DAYS,
            'min_paper_importance': MIN_PAPER_IMPORTANCE
        },
        'input_counts': {
            'active_rqs': len(active_rqs) if 'active_rqs' in globals() else 0,
            'recent_papers': len(recent_papers) if 'recent_papers' in globals() else 0,
            'recent_meetings': len(recent_meetings) if 'recent_meetings' in globals() else 0
        },
        'output_counts': {
            'proposals_generated': len(rq_proposals),
            'proposals_written': sum(1 for r in write_results if 'success' in r.get('status', '')),
            'write_failures': sum(1 for r in write_results if r.get('status') == 'error')
        },
        'artifacts': {
            'update_bundle': os.path.basename(UPDATE_BUNDLE_PATH),
            'proposals_json': os.path.basename(PROPOSALS_JSON_PATH),
            'proposals_csv': os.path.basename(PROPOSALS_CSV_PATH),
            'write_results_json': os.path.basename(write_results_path),
            'write_results_csv': os.path.basename(write_results_csv_path),
            'synthesis_json': os.path.basename(synthesis_path) if 'synthesis_result' in globals() and synthesis_result else None
        },
        'database_ids': {
            'literature': LITERATURE_DB_ID,
            'meetings': MEETING_DB_ID,
            'research_questions': RQ_DB_ID
        }
    }
    
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    
    print(f"   ✓ Saved run manifest to: {manifest_path}")
    
except Exception as e:
    print(f"   ✗ Error saving manifest: {e}")

# --- Summary report ---
print("\n" + "="*60)
print("=== Artifact Export Summary ===")
print("="*60)
print(f"\nRun timestamp: {RUN_TIMESTAMP}")
print(f"Artifact directory: {ARTIFACTS_DIR}")
print(f"\nFiles created:")

artifact_files = [
    ('Update bundle (JSON)', UPDATE_BUNDLE_PATH),
    ('Evidence bundle (JSON)', evidence_path),
    ('RQ decisions (JSON)', decisions_path),
    ('Proposals (JSON)', PROPOSALS_JSON_PATH),
    ('Write results (JSON)', write_results_path),
    ('Write results (CSV)', write_results_csv_path),
    ('Run manifest (JSON)', manifest_path),
]

for label, path in artifact_files:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"  ✓ {label}")
        print(f"    {path}")
        print(f"    Size: {size_kb:.1f} KB")
    else:
        print(f"  ⚠ {label} (not created)")

print("\n" + "="*60)
print("\n✓ All artifacts saved successfully")
print("\n=== Artifact export complete ===")
print("Next step: Run Cell 15 for summary and next steps")



=== Saving output artifacts ===
Artifact directory: ./artifacts/day24/
Run timestamp: 20260120_055814

1. Saving RQ proposals (JSON)...
   ✓ Saved 50 proposals to: ./artifacts/day24/rq_proposals_20260120_055814.json

2. Saving RQ proposals (CSV)...
   ✗ Error saving proposals CSV: 'description'

3. Saving write results (JSON)...
   ✓ Saved 32 write results to: ./artifacts/day24/write_results_20260120_055814.json

4. Saving write results (CSV)...
   ✓ Saved 32 write results to: ./artifacts/day24/write_results_20260120_055814.csv

5. Saving decision results (JSON)...
   ✓ Saved rq_decisions to: ./artifacts/day24/rq_decisions_20260120_055814.json

6. Saving evidence bundle (JSON)...
   ✓ Saved evidence_bundle to: ./artifacts/day24/evidence_bundle_20260120_055814.json

7. Saving run manifest...
   ✓ Saved run manifest to: ./artifacts/day24/run_manifest_20260120_055814.json

=== Artifact Export Summary ===

Run timestamp: 20260120_055814
Artifact directory: ./artifacts/day24/

Files create

In [60]:
# ============================================================
# Cell 15 — Summary and next steps (NEW pipeline)
# ============================================================
# Overview:
#   Run summary for the NEW pipeline:
#     evidence_bundle -> rq_decisions -> rq_proposals -> notion_payloads -> write_results
#   Avoids legacy fields (importance/priority/confidence/synthesis_result/version prop).
# Inputs:
#   active_rqs, recent_papers, recent_meetings, evidence_bundle, rq_decisions, rq_proposals,
#   notion_payloads, write_results, artifacts paths
# Outputs:
#   Console summary + actionable next steps

from collections import Counter
import os

print("\n" + "="*70)
print("="*70)
print("===                        RUN SUMMARY                        ===")
print("="*70)
print("="*70)

# --- Run Identification ---
print(f"\nRun Timestamp: {RUN_TIMESTAMP}")
print(f"Mode: {'DRY RUN (simulation only)' if DRY_RUN else 'LIVE (actual writes)'}")
try:
    print(f"LLM: {OPENAI_CONFIG.get('model','unknown')} (temp={OPENAI_CONFIG.get('temperature','?')})")
except Exception:
    pass

# --- Input Summary ---
print("\n" + "-"*70)
print("INPUT DATA SUMMARY")
print("-"*70)

if 'active_rqs' in globals():
    print(f"Active RQs (baseline):        {len(active_rqs)}")
else:
    print("Active RQs (baseline):        N/A")

if 'recent_papers' in globals():
    print(f"Recent papers (last {RECENT_PAPERS_DAYS}d):     {len(recent_papers)}")
else:
    print(f"Recent papers (last {RECENT_PAPERS_DAYS}d):     N/A")

if 'recent_meetings' in globals():
    print(f"Recent meetings (last {RECENT_MEETINGS_DAYS}d):   {len(recent_meetings)}")
else:
    print(f"Recent meetings (last {RECENT_MEETINGS_DAYS}d):   N/A")

# evidence_bundle counts
if 'evidence_bundle' in globals() and evidence_bundle:
    print(f"Evidence bundle:              papers={len(evidence_bundle.get('papers',[]))}, meetings={len(evidence_bundle.get('meetings',[]))}")
else:
    print("Evidence bundle:              N/A")

# --- Decision Summary ---
print("\n" + "-"*70)
print("RQ DECISIONS (LLM)")
print("-"*70)

if 'rq_decisions' in globals() and rq_decisions:
    decision_counts = Counter([(d.get("decision") or "UNKNOWN").upper() for d in rq_decisions])
    total_new = sum(len(d.get("new_rq_proposals") or []) for d in rq_decisions)

    print(f"RQ decisions produced:        {len(rq_decisions)}")
    for k, v in sorted(decision_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {k:10}: {v}")
    print(f"New RQ proposals (total):     {total_new}")

    # Evidence coverage check
    with_any_evidence = 0
    for d in rq_decisions:
        ev = d.get("evidence") or {}
        if (ev.get("paper_ids") or []) or (ev.get("meeting_ids") or []):
            with_any_evidence += 1
    print(f"Decisions w/ evidence:        {with_any_evidence}/{len(rq_decisions)}")
else:
    print("RQ decisions:                 N/A (not generated)")

# --- Proposals Summary ---
print("\n" + "-"*70)
print("RQ PROPOSALS (for Notion)")
print("-"*70)

if 'rq_proposals' in globals() and rq_proposals:
    action_counts = Counter([(p.get("action") or "unknown").lower() for p in rq_proposals])
    print(f"Proposals generated:          {len(rq_proposals)}")
    for k, v in sorted(action_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {k:10}: {v}")

    # Evidence distribution
    ev_papers = []
    ev_meetings = []
    for p in rq_proposals:
        ev = p.get("evidence") or {}
        ev_papers.append(len(ev.get("paper_ids") or []))
        ev_meetings.append(len(ev.get("meeting_ids") or []))
    if ev_papers:
        print(f"Evidence per proposal:        papers avg={sum(ev_papers)/len(ev_papers):.2f}, meetings avg={sum(ev_meetings)/len(ev_meetings):.2f}")

    # Show a few updates/creates
    updates = [p for p in rq_proposals if p.get("action") == "update"]
    creates = [p for p in rq_proposals if p.get("action") == "create"]
    if updates:
        print("\nSample UPDATE proposals:")
        for p in updates[:3]:
            print(f"  - {p.get('title','')[:90]}")
    if creates:
        print("\nSample CREATE proposals:")
        for p in creates[:3]:
            print(f"  - {p.get('title','')[:90]}")
else:
    print("Proposals generated:          N/A (not built)")

# --- Notion Payloads Summary ---
print("\n" + "-"*70)
print("NOTION PAYLOADS")
print("-"*70)

if 'notion_payloads' in globals() and notion_payloads:
    payload_actions = Counter([p.get("action","unknown") for p in notion_payloads])
    print(f"Payloads prepared:            {len(notion_payloads)}")
    for k, v in sorted(payload_actions.items(), key=lambda x: x[1], reverse=True):
        print(f"  {k:10}: {v}")
else:
    print("Payloads prepared:            N/A")

# --- Write Results Summary ---
print("\n" + "-"*70)
print("NOTION WRITE-BACK RESULTS")
print("-"*70)

if 'write_results' in globals() and write_results:
    total_attempts = len(write_results)
    successes = sum(1 for r in write_results if 'success' in (r.get('status') or ''))
    failures = sum(1 for r in write_results if (r.get('status') == 'error'))
    print(f"Total write attempts:         {total_attempts}")
    print(f"Successful:                   {successes}")
    print(f"Failed:                       {failures}")
    if total_attempts:
        print(f"Success rate:                 {(successes/total_attempts)*100:.1f}%")
    if DRY_RUN:
        print("\n⚠ DRY RUN MODE: No actual writes to Notion performed")
else:
    print("Write results:                N/A (Cell 13 not executed)")

# --- Artifacts Summary (uses your updated artifact_files if available) ---
print("\n" + "-"*70)
print("ARTIFACTS EXPORTED")
print("-"*70)

if 'artifact_files' in globals():
    artifact_list = artifact_files
else:
    artifact_list = []

if artifact_list:
    created = 0
    for label, path in artifact_list:
        if path and os.path.exists(path):
            created += 1
            size_kb = os.path.getsize(path) / 1024
            print(f"✓ {label:25} ({size_kb:6.1f} KB)")
        else:
            print(f"⚠ {label:25} (not created)")
    print(f"\nArtifacts created:            {created}/{len(artifact_list)}")
else:
    print("Artifacts:                    (artifact_files not defined)")

print("\n" + "-"*70)
print("RECOMMENDED NEXT STEPS")
print("-"*70)

steps = []
if DRY_RUN:
    steps.append("1) Review rq_proposals preview (Cell 12) and exported CSV/JSON artifacts.")
    steps.append("2) If OK, set DRY_RUN=False and re-run Cell 13 (write-back).")
else:
    steps.append("1) Review created Draft pages in Notion and edit/merge duplicates.")
    steps.append("2) Promote approved Drafts to Active; keep rejected ones as Draft or archive.")

steps.append("3) If too many CREATE proposals: add dedup/merge step (cluster titles by similarity) before write-back.")
steps.append("4) If decisions have weak evidence: tighten evidence selection per RQ (max_papers, add tag matching, require meeting notes).")
steps.append("5) Consider adding an explicit Version property or Parent relation to the RQ DB for cleaner version tracking.")

for s in steps:
    print(s)

print("\n" + "="*70)
print("✓ Notebook execution complete")
print("=== End of 024_rq_update_from_updates_and_meetings ===")



===                        RUN SUMMARY                        ===

Run Timestamp: 20260120_055814
Mode: LIVE (actual writes)
LLM: gpt-4o-mini (temp=0.0)

----------------------------------------------------------------------
INPUT DATA SUMMARY
----------------------------------------------------------------------
Active RQs (baseline):        30
Recent papers (last 30d):     33
Recent meetings (last 14d):   1
Evidence bundle:              papers=33, meetings=1

----------------------------------------------------------------------
RQ DECISIONS (LLM)
----------------------------------------------------------------------
RQ decisions produced:        30
  KEEP      : 18
  UPDATE    : 12
New RQ proposals (total):     20
Decisions w/ evidence:        30/30

----------------------------------------------------------------------
RQ PROPOSALS (for Notion)
----------------------------------------------------------------------
Proposals generated:          50
  create    : 20
  keep      : 18
